In [1]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/023/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_cienciapolitica.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/023/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 372 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_cienciapolitica.csv
   372 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          313
ALCANZÓ VACANTE           56
INHABILITADO (Art. 5)      2
AUSENTE                    1

── Estadísticas de puntaje ──
count     371.000
mean      896.310
std       167.461
min       434.875
25%       778.438
50%       902.500
75%      1011.625
max      1330.125


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,739314,"Aburto Chavez, Verenisce Bilha",CIENCIA POLÍTICA,774.625,NaN,SIN OBSERVACIÓN,CIENCIA POLÍTICA,https://admision.unmsm.edu.pe/Website20262/A/0...
1,696050,"Acharte Laura, Angela Luz",CIENCIA POLÍTICA,713.500,NaN,SIN OBSERVACIÓN,CIENCIA POLÍTICA,https://admision.unmsm.edu.pe/Website20262/A/0...
2,723870,"Aguilar Salazar, Walter Fabián",CIENCIA POLÍTICA,627.625,NaN,SIN OBSERVACIÓN,CIENCIA POLÍTICA,https://admision.unmsm.edu.pe/Website20262/A/0...
3,826969,"Aguilera Cortez, Aslhy Brigitte",CIENCIA POLÍTICA,838.000,NaN,SIN OBSERVACIÓN,CIENCIA POLÍTICA,https://admision.unmsm.edu.pe/Website20262/A/0...
4,762581,"Aguirre Valdez, Ariana Paola",CIENCIA POLÍTICA,1007.500,NaN,SIN OBSERVACIÓN,CIENCIA POLÍTICA,https://admision.unmsm.edu.pe/Website20262/A/0...
5,813783,"Agurto Garcia, Hallibeth Jeissy",CIENCIA POLÍTICA,731.625,NaN,SIN OBSERVACIÓN,CIENCIA POLÍTICA,https://admision.unmsm.edu.pe/Website20262/A/0...
6,825822,"Alarcon Muñoz, Sebastián José",CIENCIA POLÍTICA,669.625,NaN,SIN OBSERVACIÓN,CIENCIA POLÍTICA,https://admision.unmsm.edu.pe/Website20262/A/0...
7,791271,"Alcalá Vargas, Nicole Aurora",CIENCIA POLÍTICA,434.875,NaN,SIN OBSERVACIÓN,CIENCIA POLÍTICA,https://admision.unmsm.edu.pe/Website20262/A/0...
8,685412,"Alegre Lopez, Enzo Ruben",CIENCIA POLÍTICA,839.250,NaN,SIN OBSERVACIÓN,CIENCIA POLÍTICA,https://admision.unmsm.edu.pe/Website20262/A/0...
9,711946,"Allain Ramos, Milagros Esther",CIENCIA POLÍTICA,810.125,NaN,SIN OBSERVACIÓN,CIENCIA POLÍTICA,https://admision.unmsm.edu.pe/Website20262/A/0...


In [2]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_cienciapolitica.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: CIENCIA POLÍTICA
Vacantes obtenidas: 56
Puntaje máximo: 1330.125
Puntaje mínimo: 1070.375


,codigo,apellidos_nombres,puntaje,merito_ep
0,693125,"Prieto Alvarado, Mell Rose",1330.125,1.0
1,809215,"Jamanca Ojeda, Laura Rosenda",1320.000,2.0
2,690588,"Caira Bautista, Roxana Giomaira",1314.125,3.0
3,750185,"Andueza Espinoza, Williams Farel",1285.000,4.0
4,742515,"Bendezu Chipana, Andre Leonel",1283.875,5.0
5,751270,"Fernandez Reyes, Gabriela Elizabeth",1274.875,6.0
6,832339,"Torres Vilcabana, Diego",1265.750,7.0
7,727663,"Ñaupari Maldonado, Benjamin Isaac",1224.625,8.0
8,778810,"Cortez Maldonado, Daniela Fernanda",1223.500,9.0
9,771479,"Chamorro Enriquez, Angel Wilians",1206.375,10.0


In [3]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/101/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_biologia.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/101/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 187 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_biologia.csv
   187 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          140
ALCANZÓ VACANTE           45
AUSENTE                    1
INHABILITADO (Art. 5)      1

── Estadísticas de puntaje ──
count     186.000
mean      853.335
std       161.680
min       403.125
25%       742.250
50%       839.250
75%       958.562
max      1277.750


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,218537,"Acquaviva Romero, Francesca Lucciana",CIENCIAS BIOLÓGICAS,1021.250,31.0,ALCANZÓ VACANTE,CIENCIAS BIOLÓGICAS,https://admision.unmsm.edu.pe/Website20262/A/1...
1,215726,"Alania Espejo, Emili Angheli",CIENCIAS BIOLÓGICAS,919.750,NaN,SIN OBSERVACIÓN,CIENCIAS BIOLÓGICAS,https://admision.unmsm.edu.pe/Website20262/A/1...
2,210532,"Alaya Talavera, Zumiko Satomy Anthonella",CIENCIAS BIOLÓGICAS,758.625,NaN,SIN OBSERVACIÓN,CIENCIAS BIOLÓGICAS,https://admision.unmsm.edu.pe/Website20262/A/1...
3,211905,"Alegre Martinez, Matias Sebastian",CIENCIAS BIOLÓGICAS,843.500,NaN,SIN OBSERVACIÓN,CIENCIAS BIOLÓGICAS,https://admision.unmsm.edu.pe/Website20262/A/1...
4,219950,"Alegria Garcia, Jhosep Smith",CIENCIAS BIOLÓGICAS,692.750,NaN,SIN OBSERVACIÓN,CIENCIAS BIOLÓGICAS,https://admision.unmsm.edu.pe/Website20262/A/1...
5,210578,"Altamirano Laveriano, Alexis Aimar",CIENCIAS BIOLÓGICAS,1016.250,33.0,ALCANZÓ VACANTE,CIENCIAS BIOLÓGICAS,https://admision.unmsm.edu.pe/Website20262/A/1...
6,216868,"Altamirano Salvador, Arely Andrea",CIENCIAS BIOLÓGICAS,614.875,NaN,SIN OBSERVACIÓN,CIENCIAS BIOLÓGICAS,https://admision.unmsm.edu.pe/Website20262/A/1...
7,218433,"Amaut Bendezu, Jestyn Braison",CIENCIAS BIOLÓGICAS,759.875,NaN,SIN OBSERVACIÓN,CIENCIAS BIOLÓGICAS,https://admision.unmsm.edu.pe/Website20262/A/1...
8,211016,"Ambicho Oscco, Alejandro David",CIENCIAS BIOLÓGICAS,941.375,NaN,SIN OBSERVACIÓN,CIENCIAS BIOLÓGICAS,https://admision.unmsm.edu.pe/Website20262/A/1...
9,218136,"Ambrosio Silva, Roberto David",CIENCIAS BIOLÓGICAS,754.125,NaN,SIN OBSERVACIÓN,CIENCIAS BIOLÓGICAS,https://admision.unmsm.edu.pe/Website20262/A/1...


In [4]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_biologia.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: CIENCIAS BIOLÓGICAS
Vacantes obtenidas: 45
Puntaje máximo: 1277.75
Puntaje mínimo: 964.125


,codigo,apellidos_nombres,puntaje,merito_ep
0,212822,"Cieza Chávez, Adrián Asad",1277.750,1.0
1,216177,"Souza Sabogal, Benjamín Daniel",1265.750,2.0
2,216052,"Caruajulca Eugenio, Noemi Abigail",1253.500,3.0
3,213686,"Oliva Leon, Abhryl Luana",1248.000,4.0
4,211173,"Torpoco Capcha, José Manuel",1210.375,5.0
5,214120,"Soto Zumaran, Alexa Thais",1193.250,6.0
6,213250,"Zegarra Mamani, Enzo Alonso",1155.500,7.0
7,215954,"Quiroz Valenzuela, Maria Alejandra",1150.875,8.0
8,211482,"Andia Arce, Rosalin",1132.125,9.0
9,210974,"Cruz Condori, Marco Antonio Ernesto",1131.000,10.0


In [5]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/145/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_computacioncientifica.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/145/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 42 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_computacioncientifica.csv
   42 filas  ×  8 columnas

── Observaciones ──
observacion
ALCANZÓ VACANTE    42

── Estadísticas de puntaje ──
count      42.000
mean      770.717
std       137.794
min       576.000
25%       657.781
50%       760.375
75%       858.375
max      1054.500


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,219773,"Alarcon Atauje, Ezzio Cesar Manases",COMPUTACIÓN CIENTÍFICA,860.375,11,ALCANZÓ VACANTE,COMPUTACIÓN CIENTÍFICA,https://admision.unmsm.edu.pe/Website20262/A/1...
1,217075,"Amaya Aquino, Solanghs Julient Milagros",COMPUTACIÓN CIENTÍFICA,908.125,7,ALCANZÓ VACANTE,COMPUTACIÓN CIENTÍFICA,https://admision.unmsm.edu.pe/Website20262/A/1...
2,217935,"Ambrocio Estela, Edinson Francisco",COMPUTACIÓN CIENTÍFICA,1020.250,2,ALCANZÓ VACANTE,COMPUTACIÓN CIENTÍFICA,https://admision.unmsm.edu.pe/Website20262/A/1...
3,218732,"Atto Salhua, Yinnsu Satomy",COMPUTACIÓN CIENTÍFICA,578.125,42,ALCANZÓ VACANTE,COMPUTACIÓN CIENTÍFICA,https://admision.unmsm.edu.pe/Website20262/A/1...
4,217838,"Barbadillo Silva, Cristoper Junior",COMPUTACIÓN CIENTÍFICA,826.125,15,ALCANZÓ VACANTE,COMPUTACIÓN CIENTÍFICA,https://admision.unmsm.edu.pe/Website20262/A/1...
5,213692,"Beltran Castillo, Miguel Angel",COMPUTACIÓN CIENTÍFICA,609.875,37,ALCANZÓ VACANTE,COMPUTACIÓN CIENTÍFICA,https://admision.unmsm.edu.pe/Website20262/A/1...
6,215948,"Benites Olivos, Leonardo Fabricio",COMPUTACIÓN CIENTÍFICA,678.250,30,ALCANZÓ VACANTE,COMPUTACIÓN CIENTÍFICA,https://admision.unmsm.edu.pe/Website20262/A/1...
7,214405,"Bravo Espinoza, Max Eduar",COMPUTACIÓN CIENTÍFICA,809.000,17,ALCANZÓ VACANTE,COMPUTACIÓN CIENTÍFICA,https://admision.unmsm.edu.pe/Website20262/A/1...
8,210755,"Caceres Sarmiento, Milagros Sunmy",COMPUTACIÓN CIENTÍFICA,758.750,22,ALCANZÓ VACANTE,COMPUTACIÓN CIENTÍFICA,https://admision.unmsm.edu.pe/Website20262/A/1...
9,219598,"Cajas Aquino, Luis Alvaro",COMPUTACIÓN CIENTÍFICA,1020.250,3,ALCANZÓ VACANTE,COMPUTACIÓN CIENTÍFICA,https://admision.unmsm.edu.pe/Website20262/A/1...


In [6]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_computacioncientifica.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: COMPUTACIÓN CIENTÍFICA
Vacantes obtenidas: 42
Puntaje máximo: 1054.5
Puntaje mínimo: 576.0


,codigo,apellidos_nombres,puntaje,merito_ep
0,216335,"De La Cruz Silva, Steven",1054.500,1
1,219598,"Cajas Aquino, Luis Alvaro",1020.250,3
2,217935,"Ambrocio Estela, Edinson Francisco",1020.250,2
3,218419,"Maldonado Oré, Javier Shande",1009.375,4
4,213306,"Heredia Carranza, Joycelyn",979.625,5
5,218332,"Villon Barrientos, Fabiana Valeria",952.875,6
6,217075,"Amaya Aquino, Solanghs Julient Milagros",908.125,7
7,216818,"Santos Salazar, Angel Berlin",903.750,8
8,216521,"Moreno Cardenas, Felipe Hilario",900.375,9
9,217917,"Paccochuco Herrera, Nelson Abel",895.750,10


In [7]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/035/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_comu.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/035/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 301 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_comu.csv
   301 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          266
ALCANZÓ VACANTE           32
AUSENTE                    2
INHABILITADO (Art. 5)      1

── Estadísticas de puntaje ──
count     299.000
mean      848.842
std       189.064
min       290.750
25%       721.688
50%       851.875
75%       986.625
max      1325.125


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,797795,"Acosta Huamani, Azalea Sady",COMUNICACIÓN SOCIAL,1245.750,5.0,ALCANZÓ VACANTE,COMUNICACIÓN SOCIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
1,741684,"Acuña Ramirez, Gabriela",COMUNICACIÓN SOCIAL,758.125,NaN,SIN OBSERVACIÓN,COMUNICACIÓN SOCIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
2,689172,"Aguilar Ramirez, Victoria Gabriela",COMUNICACIÓN SOCIAL,573.250,NaN,SIN OBSERVACIÓN,COMUNICACIÓN SOCIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
3,707044,"Aguirre Lozano, Alisson Jimena",COMUNICACIÓN SOCIAL,985.375,NaN,SIN OBSERVACIÓN,COMUNICACIÓN SOCIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
4,792461,"Alberto Valiente, Katia Marlene",COMUNICACIÓN SOCIAL,864.375,NaN,SIN OBSERVACIÓN,COMUNICACIÓN SOCIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
5,789751,"Alcala Sanchez, Sofia Alejanddra",COMUNICACIÓN SOCIAL,692.125,NaN,SIN OBSERVACIÓN,COMUNICACIÓN SOCIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
6,691250,"Alcalá Torres, Tatiana Valeria",COMUNICACIÓN SOCIAL,1148.125,23.0,ALCANZÓ VACANTE,COMUNICACIÓN SOCIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
7,690148,"Alcantara Galindo, Anderson Deyvis",COMUNICACIÓN SOCIAL,720.500,NaN,SIN OBSERVACIÓN,COMUNICACIÓN SOCIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
8,798835,"Aliaga Vargas, Evan Owen",COMUNICACIÓN SOCIAL,670.500,NaN,SIN OBSERVACIÓN,COMUNICACIÓN SOCIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
9,738148,"Allcarima Libia, Nayely Kiara",COMUNICACIÓN SOCIAL,869.500,NaN,SIN OBSERVACIÓN,COMUNICACIÓN SOCIAL,https://admision.unmsm.edu.pe/Website20262/A/0...


In [8]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_comu.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: COMUNICACIÓN SOCIAL
Vacantes obtenidas: 32
Puntaje máximo: 1325.125
Puntaje mínimo: 1099.0


,codigo,apellidos_nombres,puntaje,merito_ep
0,819022,"Collahua Torres, Josue Moises",1325.125,1.0
1,694160,"Cuyubamba Coronel, David Elias",1294.875,2.0
2,702226,"Villanueva Flores, Angelo Junior",1294.875,3.0
3,750373,"Quiroz Ronceros, David Antonio",1264.000,4.0
4,797795,"Acosta Huamani, Azalea Sady",1245.750,5.0
5,833133,"Carrillo Mejia, Jazmin Milagros",1242.750,6.0
6,779653,"Espinoza Laime, Sebastian Andree",1223.500,7.0
7,800500,"Gutierrez Colan, Diana Lizbeth Mia",1210.375,8.0
8,695798,"Huillca Rojas, Estefany Isabel",1210.375,9.0
9,814540,"Soto Yalo, Sharon Marciel",1209.000,10.0


In [9]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/039/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_conservacion.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/039/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 39 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_conservacion.csv
   39 filas  ×  8 columnas

── Observaciones ──
observacion
ALCANZÓ VACANTE    28
SIN OBSERVACIÓN    11

── Estadísticas de puntaje ──
count      39.000
mean      789.965
std       138.195
min       529.875
25%       692.250
50%       793.250
75%       869.500
max      1075.625


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,754765,"Arevalo Huaman, Steven",CONSERVACIÓN Y RESTAURACIÓN,988.250,4.0,ALCANZÓ VACANTE,CONSERVACIÓN Y RESTAURACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
1,712288,"Arones Allaucca, Jimena Anyeli",CONSERVACIÓN Y RESTAURACIÓN,916.125,6.0,ALCANZÓ VACANTE,CONSERVACIÓN Y RESTAURACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
2,769954,"Beizaga Suarez, Luis Andres",CONSERVACIÓN Y RESTAURACIÓN,970.250,5.0,ALCANZÓ VACANTE,CONSERVACIÓN Y RESTAURACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
3,690104,"Calderon Estumbelo, Glorymar",CONSERVACIÓN Y RESTAURACIÓN,657.125,NaN,SIN OBSERVACIÓN,CONSERVACIÓN Y RESTAURACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
4,812801,"Camacho Leiva, Luis Alexander",CONSERVACIÓN Y RESTAURACIÓN,794.125,19.0,ALCANZÓ VACANTE,CONSERVACIÓN Y RESTAURACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
5,779079,"Castillo Sanchez, Fabricio Alexander",CONSERVACIÓN Y RESTAURACIÓN,691.500,NaN,SIN OBSERVACIÓN,CONSERVACIÓN Y RESTAURACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
6,769349,"Ccanre Taya, Anahi Magdiel",CONSERVACIÓN Y RESTAURACIÓN,793.250,20.0,ALCANZÓ VACANTE,CONSERVACIÓN Y RESTAURACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
7,687544,"Condori Cornejo, Fabricio Aldahier",CONSERVACIÓN Y RESTAURACIÓN,771.875,22.0,ALCANZÓ VACANTE,CONSERVACIÓN Y RESTAURACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
8,741925,"Díaz Olivera, Cielo Mileny",CONSERVACIÓN Y RESTAURACIÓN,855.375,12.0,ALCANZÓ VACANTE,CONSERVACIÓN Y RESTAURACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...
9,685829,"Garcia Cabrera, Oriele Alejandra",CONSERVACIÓN Y RESTAURACIÓN,529.875,NaN,SIN OBSERVACIÓN,CONSERVACIÓN Y RESTAURACIÓN,https://admision.unmsm.edu.pe/Website20262/A/0...


In [10]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_conservacion.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: CONSERVACIÓN Y RESTAURACIÓN
Vacantes obtenidas: 28
Puntaje máximo: 1075.625
Puntaje mínimo: 697.625


,codigo,apellidos_nombres,puntaje,merito_ep
0,734694,"Torres Melendez, Sebastian Isaac",1075.625,1.0
1,687985,"Villagaray Trejo, Mariajose Estilita",1075.000,2.0
2,781476,"Mariluz Aguilar, Isabella Silvana",1045.375,3.0
3,754765,"Arevalo Huaman, Steven",988.250,4.0
4,769954,"Beizaga Suarez, Luis Andres",970.250,5.0
5,712288,"Arones Allaucca, Jimena Anyeli",916.125,6.0
6,707984,"Mesias Perez, Darla Grecia Hazel",911.750,7.0
7,800915,"Sanchez Tarazona, Rosa Jazmin",910.625,8.0
8,694731,"Rojas Arguelles, Jacory Jimena",886.125,9.0
9,817530,"Suarez Anaya, Diego David",882.625,10.0


In [11]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/111/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_contabilidad.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/111/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 479 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_contabilidad.csv
   479 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          309
ALCANZÓ VACANTE          165
AUSENTE                    4
INHABILITADO (Art. 5)      1

── Estadísticas de puntaje ──
count     475.000
mean      816.174
std       175.550
min       315.250
25%       700.500
50%       810.125
75%       940.875
max      1359.375


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,590867,"Abanto Salazar, Maria Del Carmen",CONTABILIDAD,840.250,NaN,SIN OBSERVACIÓN,CONTABILIDAD,https://admision.unmsm.edu.pe/Website20262/A/1...
1,579553,"Acevedo Huallpa, Juluan Anthony",CONTABILIDAD,542.250,NaN,SIN OBSERVACIÓN,CONTABILIDAD,https://admision.unmsm.edu.pe/Website20262/A/1...
2,654278,"Acosta Riva, Fernanda Denisse",CONTABILIDAD,787.875,NaN,SIN OBSERVACIÓN,CONTABILIDAD,https://admision.unmsm.edu.pe/Website20262/A/1...
3,657451,"Acuña Bolaños, Anai Antuane",CONTABILIDAD,742.125,NaN,SIN OBSERVACIÓN,CONTABILIDAD,https://admision.unmsm.edu.pe/Website20262/A/1...
4,648430,"Aguero Chuica, Ellery Fabian",CONTABILIDAD,982.000,79.0,ALCANZÓ VACANTE,CONTABILIDAD,https://admision.unmsm.edu.pe/Website20262/A/1...
5,649196,"Aguilar Condori, Damaris Ariasu",CONTABILIDAD,960.000,98.0,ALCANZÓ VACANTE,CONTABILIDAD,https://admision.unmsm.edu.pe/Website20262/A/1...
6,647227,"Aguilar Condori, Dayana Milagros",CONTABILIDAD,1039.375,47.0,ALCANZÓ VACANTE,CONTABILIDAD,https://admision.unmsm.edu.pe/Website20262/A/1...
7,658833,"Alarcon Asturayme, Yadhira Anabel",CONTABILIDAD,855.000,NaN,SIN OBSERVACIÓN,CONTABILIDAD,https://admision.unmsm.edu.pe/Website20262/A/1...
8,650517,"Alarcon Chaupiz, Maria Del Carmen",CONTABILIDAD,1023.375,59.0,ALCANZÓ VACANTE,CONTABILIDAD,https://admision.unmsm.edu.pe/Website20262/A/1...
9,560922,"Alberto Flores, Jaime Daniel",CONTABILIDAD,779.875,NaN,SIN OBSERVACIÓN,CONTABILIDAD,https://admision.unmsm.edu.pe/Website20262/A/1...


In [12]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_contabilidad.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: CONTABILIDAD
Vacantes obtenidas: 165
Puntaje máximo: 1359.375
Puntaje mínimo: 876.375


,codigo,apellidos_nombres,puntaje,merito_ep
0,646264,"Luna Asian, Fatima Katyuska",1359.375,1.0
1,568722,"Gomez Medina, Anggela Mariaelena",1321.125,2.0
2,653507,"Cabezas Arango, Miguel Gesebt",1300.000,3.0
3,656081,"Guillen Bramon, Joe Leonardo",1290.875,4.0
4,650778,"Huaranga Gonzales, Anggela Ruby",1272.000,5.0
...,...,...,...,...
160,654440,"Mio Najera, David Felix",880.000,161.0
161,592897,"Rodriguez Fernández, Carlos",879.375,162.0
162,651356,"Rodriguez Ruiz, Ronaile Valentino",878.875,163.0
163,588224,"Davila Valencia, Keysi Maritza",877.500,164.0


In [13]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/115/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_criminalistica.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/115/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 168 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_criminalistica.csv
   168 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          121
ALCANZÓ VACANTE           45
AUSENTE                    1
INHABILITADO (Art. 5)      1

── Estadísticas de puntaje ──
count     167.000
mean      777.687
std       149.072
min       453.875
25%       668.312
50%       770.750
75%       855.750
max      1132.125


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,652518,"Abregu Tipacti, Anibal Guillermo",CRIMINALÍSTICA FINANCIERA FORENSE,NaN,NaN,AUSENTE,CRIMINALÍSTICA FINANCIERA FORENSE,https://admision.unmsm.edu.pe/Website20262/A/1...
1,588198,"Aguilar Garcia, Karol Camila",CRIMINALÍSTICA FINANCIERA FORENSE,775.875,NaN,SIN OBSERVACIÓN,CRIMINALÍSTICA FINANCIERA FORENSE,https://admision.unmsm.edu.pe/Website20262/A/1...
2,650615,"Aguilar Paucar, Fabricio Leandro",CRIMINALÍSTICA FINANCIERA FORENSE,810.125,NaN,SIN OBSERVACIÓN,CRIMINALÍSTICA FINANCIERA FORENSE,https://admision.unmsm.edu.pe/Website20262/A/1...
3,652607,"Aguilar Zamora, Mateo Enrique",CRIMINALÍSTICA FINANCIERA FORENSE,641.125,NaN,SIN OBSERVACIÓN,CRIMINALÍSTICA FINANCIERA FORENSE,https://admision.unmsm.edu.pe/Website20262/A/1...
4,554818,"Agurto Rosario, Bily Esmit",CRIMINALÍSTICA FINANCIERA FORENSE,623.375,NaN,SIN OBSERVACIÓN,CRIMINALÍSTICA FINANCIERA FORENSE,https://admision.unmsm.edu.pe/Website20262/A/1...
5,551546,"Almeyda Saravia, Cinthya Mariela",CRIMINALÍSTICA FINANCIERA FORENSE,862.000,39.0,ALCANZÓ VACANTE,CRIMINALÍSTICA FINANCIERA FORENSE,https://admision.unmsm.edu.pe/Website20262/A/1...
6,645800,"Alvarado Mamani, Dulce Luciana Thalia",CRIMINALÍSTICA FINANCIERA FORENSE,835.250,NaN,SIN OBSERVACIÓN,CRIMINALÍSTICA FINANCIERA FORENSE,https://admision.unmsm.edu.pe/Website20262/A/1...
7,592397,"Alvarez Melchor, Lucero Leonela",CRIMINALÍSTICA FINANCIERA FORENSE,726.750,NaN,SIN OBSERVACIÓN,CRIMINALÍSTICA FINANCIERA FORENSE,https://admision.unmsm.edu.pe/Website20262/A/1...
8,649427,"Alvarez Tocto, Gustavo Fernando",CRIMINALÍSTICA FINANCIERA FORENSE,708.500,NaN,SIN OBSERVACIÓN,CRIMINALÍSTICA FINANCIERA FORENSE,https://admision.unmsm.edu.pe/Website20262/A/1...
9,570760,"Amoroto Rojas, Freyhtzzy Yanely",CRIMINALÍSTICA FINANCIERA FORENSE,682.500,NaN,SIN OBSERVACIÓN,CRIMINALÍSTICA FINANCIERA FORENSE,https://admision.unmsm.edu.pe/Website20262/A/1...


In [15]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_criminalistica.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: CRIMINALÍSTICA FINANCIERA FORENSE
Vacantes obtenidas: 45
Puntaje máximo: 1132.125
Puntaje mínimo: 854.625


,codigo,apellidos_nombres,puntaje,merito_ep
0,649566,"Zamudio Reyes, Xiomara Geraldine",1132.125,1.0
1,656523,"Cardenas Parian, Alexia Mariel",1119.000,2.0
2,568840,"Garcia Asturay, Eva Ariana",1097.625,3.0
3,575886,"Aybar Mitac, Lionel Salvador",1090.875,4.0
4,653123,"Vargas Requena, Fabricio Gabriel",1069.500,5.0
5,646616,"Manrique Flores, Marcel Ramiro",1062.500,6.0
6,655345,"Zelaya Navarro, Frank Xavier",1062.500,7.0
7,590303,"Nolasco Ojeda, Heisser",1058.500,8.0
8,576607,"Castañeda Quillo, Marco Antonio Urbano",1057.250,9.0
9,657370,"Montano Coronel, Leonor Del Carmen",1040.250,10.0


In [16]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/022/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_derecho.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/022/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 1873 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_derecho.csv
   1873 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          1689
ALCANZÓ VACANTE           140
INHABILITADO (Art. 5)      25
AUSENTE                    19

── Estadísticas de puntaje ──
count    1854.000
mean      888.031
std       204.145
min       148.625
25%       739.250
50%       886.875
75%      1031.531
max      1638.000


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,728518,"Abad Tabara, Valery Nicole",DERECHO,912.625,NaN,SIN OBSERVACIÓN,DERECHO,https://admision.unmsm.edu.pe/Website20262/A/0...
1,709843,"Abado Cordova, Lin Wendy Viviana",DERECHO,726.750,NaN,SIN OBSERVACIÓN,DERECHO,https://admision.unmsm.edu.pe/Website20262/A/0...
2,763699,"Acevedo Rojas, Jade Samira",DERECHO,1198.375,119.0,ALCANZÓ VACANTE,DERECHO,https://admision.unmsm.edu.pe/Website20262/A/0...
3,698962,"Acosta Estrella, Adriana Patricia",DERECHO,803.375,NaN,SIN OBSERVACIÓN,DERECHO,https://admision.unmsm.edu.pe/Website20262/A/0...
4,745347,"Acosta Lopez, Fabrizio Sebastian",DERECHO,1003.125,NaN,SIN OBSERVACIÓN,DERECHO,https://admision.unmsm.edu.pe/Website20262/A/0...
5,756099,"Acuña Vargas, Richard Alex",DERECHO,1504.375,2.0,ALCANZÓ VACANTE,DERECHO,https://admision.unmsm.edu.pe/Website20262/A/0...
6,703742,"Aguedo Cirilo, Rodrigo Lucas",DERECHO,1300.000,37.0,ALCANZÓ VACANTE,DERECHO,https://admision.unmsm.edu.pe/Website20262/A/0...
7,760490,"Aguilar Facundo, Leandro Enrique",DERECHO,975.125,NaN,SIN OBSERVACIÓN,DERECHO,https://admision.unmsm.edu.pe/Website20262/A/0...
8,762286,"Aguilar Matos, Luhanna Aydee Pamela",DERECHO,655.250,NaN,SIN OBSERVACIÓN,DERECHO,https://admision.unmsm.edu.pe/Website20262/A/0...
9,733028,"Aguilar Montoya, Mirella",DERECHO,935.625,NaN,SIN OBSERVACIÓN,DERECHO,https://admision.unmsm.edu.pe/Website20262/A/0...


In [17]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_derecho.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: DERECHO
Vacantes obtenidas: 140
Puntaje máximo: 1638.0
Puntaje mínimo: 1186.375


,codigo,apellidos_nombres,puntaje,merito_ep
0,824014,"Espinoza Rojas, Gino Daniel",1638.000,1.0
1,756099,"Acuña Vargas, Richard Alex",1504.375,2.0
2,831946,"Minaya Serra, Solange Christell",1504.375,3.0
3,776466,"Rosales Cuya, Ricardo Sebastian",1502.125,4.0
4,800587,"Au Yeung Zevallos, Angelo Fabrizio",1485.000,5.0
...,...,...,...,...
135,842786,"Cosco Wilcamango, Nadya Sarahi",1189.250,137.0
136,830748,"Llacsahuanga Cortez, Zuleyka Dayanara",1189.250,136.0
137,698624,"Rebata Ochante, Fernanda Tahis",1188.500,138.0
138,720176,"Vivar Ocsas, Luciana Noreliz",1187.500,139.0


In [18]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/121/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_economia.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/121/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 659 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_economia.csv
   659 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          565
ALCANZÓ VACANTE           79
AUSENTE                    8
INHABILITADO (Art. 5)      7

── Estadísticas de puntaje ──
count     651.000
mean      901.060
std       196.627
min       307.750
25%       758.375
50%       897.500
75%      1026.500
max      1545.500


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,654905,"Abanto Diaz, Cristofer Saulito",ECONOMÍA,956.875,NaN,SIN OBSERVACIÓN,ECONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
1,583738,"Abanto Vega, Ruben Andersson",ECONOMÍA,1202.375,49.0,ALCANZÓ VACANTE,ECONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
2,646288,"Abarca Rosas, Jhoan Pierce",ECONOMÍA,914.625,NaN,SIN OBSERVACIÓN,ECONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
3,648321,"Abele Farfán, Astrid Josefina",ECONOMÍA,832.375,NaN,SIN OBSERVACIÓN,ECONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
4,645737,"Acosta Mori, Leonela Victoria",ECONOMÍA,843.875,NaN,SIN OBSERVACIÓN,ECONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
5,579272,"Advincula Damacio, Julio Cesar",ECONOMÍA,663.375,NaN,SIN OBSERVACIÓN,ECONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
6,645236,"Afuso Morales, Luss De Los Angeles",ECONOMÍA,872.375,NaN,SIN OBSERVACIÓN,ECONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
7,555062,"Aguilar Zevallos, Lucas Samuel",ECONOMÍA,1253.125,25.0,ALCANZÓ VACANTE,ECONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
8,659343,"Aguirre Huyhua, Joshua Mathias",ECONOMÍA,1253.750,23.0,ALCANZÓ VACANTE,ECONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/1...
9,659111,"Alarcon Llactas, Jean Franco",ECONOMÍA,576.000,NaN,SIN OBSERVACIÓN,ECONOMÍA,https://admision.unmsm.edu.pe/Website20262/A/1...


In [19]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_economia.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ECONOMÍA
Vacantes obtenidas: 79
Puntaje máximo: 1545.5
Puntaje mínimo: 1148.125


,codigo,apellidos_nombres,puntaje,merito_ep
0,653467,"Heredia Mendoza, Alexander Paolo Victor",1545.500,1.0
1,551813,"Torres Pozo, Mauricio Rafael",1481.000,2.0
2,649678,"Puican Aldoradin, Rodrigo Jorge",1455.875,3.0
3,657179,"Chambilla Vilca, Frank Emmanuel",1414.750,4.0
4,651751,"Hermoza Cabellos, Johnny Antonio",1390.125,5.0
...,...,...,...,...
74,655231,"Medina Nicho, Joaquín Andrée Leonardo",1150.375,74.0
75,581267,"Docto Shica, Angel Anderson",1149.250,77.0
76,652868,"Sánchez Flores, Leonardo Gabriel",1149.250,76.0
77,564376,"Lino Jimenez, Nicole Alexandra",1148.125,79.0


In [20]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/123/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_ecointer.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/123/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 311 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_ecointer.csv
   311 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          222
ALCANZÓ VACANTE           84
INHABILITADO (Art. 5)      4
AUSENTE                    1

── Estadísticas de puntaje ──
count     310.000
mean      887.320
std       181.114
min       445.875
25%       739.844
50%       883.188
75%      1015.781
max      1405.625


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,654560,"Aguedo Loayza, Luis Adolfo De Jesús",ECONOMÍA INTERNACIONAL,962.000,NaN,SIN OBSERVACIÓN,ECONOMÍA INTERNACIONAL,https://admision.unmsm.edu.pe/Website20262/A/1...
1,651061,"Aguilar Gálvez, Edson Alexandro",ECONOMÍA INTERNACIONAL,719.250,NaN,SIN OBSERVACIÓN,ECONOMÍA INTERNACIONAL,https://admision.unmsm.edu.pe/Website20262/A/1...
2,645368,"Albinagorta Ruidias, Leonel Aaron",ECONOMÍA INTERNACIONAL,807.250,NaN,SIN OBSERVACIÓN,ECONOMÍA INTERNACIONAL,https://admision.unmsm.edu.pe/Website20262/A/1...
3,652923,"Alca Sanchez, Noemi",ECONOMÍA INTERNACIONAL,747.875,NaN,SIN OBSERVACIÓN,ECONOMÍA INTERNACIONAL,https://admision.unmsm.edu.pe/Website20262/A/1...
4,594381,"Alcala Triviños, Luis Angel",ECONOMÍA INTERNACIONAL,835.250,NaN,SIN OBSERVACIÓN,ECONOMÍA INTERNACIONAL,https://admision.unmsm.edu.pe/Website20262/A/1...
5,566689,"Ali Rojas, Andry Alexander",ECONOMÍA INTERNACIONAL,907.750,NaN,SIN OBSERVACIÓN,ECONOMÍA INTERNACIONAL,https://admision.unmsm.edu.pe/Website20262/A/1...
6,586808,"Altamiza Gonzales, Evans Exal",ECONOMÍA INTERNACIONAL,1054.500,54.0,ALCANZÓ VACANTE,ECONOMÍA INTERNACIONAL,https://admision.unmsm.edu.pe/Website20262/A/1...
7,583384,"Alva Cadillo, Arianna",ECONOMÍA INTERNACIONAL,1144.125,28.0,ALCANZÓ VACANTE,ECONOMÍA INTERNACIONAL,https://admision.unmsm.edu.pe/Website20262/A/1...
8,651434,"Alvarez Velasquez, Sebastián Imanol",ECONOMÍA INTERNACIONAL,876.875,NaN,SIN OBSERVACIÓN,ECONOMÍA INTERNACIONAL,https://admision.unmsm.edu.pe/Website20262/A/1...
9,654565,"Anaya Bojórquez, Valeri Alexandra",ECONOMÍA INTERNACIONAL,661.125,NaN,SIN OBSERVACIÓN,ECONOMÍA INTERNACIONAL,https://admision.unmsm.edu.pe/Website20262/A/1...


In [21]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_ecointer.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ECONOMÍA INTERNACIONAL
Vacantes obtenidas: 84
Puntaje máximo: 1405.625
Puntaje mínimo: 1009.375


,codigo,apellidos_nombres,puntaje,merito_ep
0,592477,"Risco Santiago, Alejandro",1405.625,1.0
1,652462,"Rojas Nario, Julio Javier",1401.000,2.0
2,653919,"Pomahuallca Guerrera, Anahi Yadhira",1306.250,3.0
3,594730,"Minaya Huacho, Sebastian Aaron",1300.000,4.0
4,658715,"Ballena Espinoza, Marcelo Alonso",1261.750,5.0
...,...,...,...,...
79,654456,"Caceres Pancorbo, Sergio Alonso",1013.250,80.0
80,659347,"Pipa Delgado, Jesús Roberto",1012.125,81.0
81,647999,"Callan Reyes, Kevin Luis",1011.000,82.0
82,648351,"Jimenez Florencio, Denis Francis",1009.375,84.0


In [22]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/122/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_ecopubli.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/122/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 174 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_ecopubli.csv
   174 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          89
ALCANZÓ VACANTE          83
INHABILITADO (Art. 5)     2

── Estadísticas de puntaje ──
count     174.000
mean      888.112
std       167.917
min       482.500
25%       770.000
50%       886.875
75%      1004.969
max      1356.375


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,649472,"Adanaque Salazar, Jostin Samir",ECONOMÍA PÚBLICA,674.250,NaN,SIN OBSERVACIÓN,ECONOMÍA PÚBLICA,https://admision.unmsm.edu.pe/Website20262/A/1...
1,659882,"Aguinaga Rodriguez, Daniela Fernanda",ECONOMÍA PÚBLICA,979.125,48.0,ALCANZÓ VACANTE,ECONOMÍA PÚBLICA,https://admision.unmsm.edu.pe/Website20262/A/1...
2,656423,"Alarcon Castro, Jenifer Yesenia",ECONOMÍA PÚBLICA,872.500,NaN,SIN OBSERVACIÓN,ECONOMÍA PÚBLICA,https://admision.unmsm.edu.pe/Website20262/A/1...
3,592996,"Alarcon Pflucker, Hanssel Alejandro",ECONOMÍA PÚBLICA,771.875,NaN,SIN OBSERVACIÓN,ECONOMÍA PÚBLICA,https://admision.unmsm.edu.pe/Website20262/A/1...
4,653918,"Aller Julca, Yameli Cristina",ECONOMÍA PÚBLICA,982.500,47.0,ALCANZÓ VACANTE,ECONOMÍA PÚBLICA,https://admision.unmsm.edu.pe/Website20262/A/1...
5,645361,"Andagua Centeno, Jesus Augusto",ECONOMÍA PÚBLICA,966.000,52.0,ALCANZÓ VACANTE,ECONOMÍA PÚBLICA,https://admision.unmsm.edu.pe/Website20262/A/1...
6,646665,"Angeles Pezo, Ailyn Sun",ECONOMÍA PÚBLICA,836.375,NaN,SIN OBSERVACIÓN,ECONOMÍA PÚBLICA,https://admision.unmsm.edu.pe/Website20262/A/1...
7,655146,"Antezana Vera, Gian Christian",ECONOMÍA PÚBLICA,1107.000,14.0,ALCANZÓ VACANTE,ECONOMÍA PÚBLICA,https://admision.unmsm.edu.pe/Website20262/A/1...
8,648826,"Anyosa Castañeda, Samuel Leonardo",ECONOMÍA PÚBLICA,876.375,NaN,SIN OBSERVACIÓN,ECONOMÍA PÚBLICA,https://admision.unmsm.edu.pe/Website20262/A/1...
9,583508,"Apfata Taipe, Josue Fernando",ECONOMÍA PÚBLICA,992.250,46.0,ALCANZÓ VACANTE,ECONOMÍA PÚBLICA,https://admision.unmsm.edu.pe/Website20262/A/1...


In [23]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_ecopubli.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ECONOMÍA PÚBLICA
Vacantes obtenidas: 83
Puntaje máximo: 1356.375
Puntaje mínimo: 900.875


,codigo,apellidos_nombres,puntaje,merito_ep
0,557551,"Vega Gonzales, Gianpool Sebastian",1356.375,1.0
1,581367,"Vasquez Corman, Wiliam Adrian",1326.250,2.0
2,570289,"Rosales Torres, Alessandro Carlo",1302.125,3.0
3,556156,"Jares Guevara, Ariel Franchesco",1257.125,4.0
4,566142,"Canales Flores, Carla Elizabeth",1224.625,5.0
...,...,...,...,...
78,650496,"Vega Castillo, Sebastian Kevin",906.625,78.0
79,653261,"Urbano Dominguez, Daniel Niles",906.625,79.0
80,654786,"Serna Vega, Gabriela Alexandra",904.875,81.0
81,652324,"Quicaño Aguirre, Mayara Adriana",900.875,82.0


In [24]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/062/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_edfisica.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/062/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 160 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_edfisica.csv
   160 filas  ×  8 columnas

── Observaciones ──
observacion
ALCANZÓ VACANTE          95
SIN OBSERVACIÓN          63
INHABILITADO (Art. 5)     1
AUSENTE                   1

── Estadísticas de puntaje ──
count     159.000
mean      934.754
std       126.880
min       595.005
25%       845.448
50%       928.967
75%      1032.438
max      1248.171


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,832308,"Abal Hilario, Armando Fabián",EDUCACIÓN FÍSICA,1050.326,32.0,ALCANZÓ VACANTE,EDUCACIÓN FÍSICA,https://admision.unmsm.edu.pe/Website20262/A/0...
1,729705,"Acevedo Baca, Alejandro Nicolas",EDUCACIÓN FÍSICA,966.942,64.0,ALCANZÓ VACANTE,EDUCACIÓN FÍSICA,https://admision.unmsm.edu.pe/Website20262/A/0...
2,829451,"Agurto Salazar, Angelo Abraham",EDUCACIÓN FÍSICA,871.550,NaN,SIN OBSERVACIÓN,EDUCACIÓN FÍSICA,https://admision.unmsm.edu.pe/Website20262/A/0...
3,817144,"Alania Atavillos, Edilson Roger",EDUCACIÓN FÍSICA,707.205,NaN,SIN OBSERVACIÓN,EDUCACIÓN FÍSICA,https://admision.unmsm.edu.pe/Website20262/A/0...
4,829672,"Alavena Eyzaguirre, Stefano Rolando",EDUCACIÓN FÍSICA,856.121,NaN,SIN OBSERVACIÓN,EDUCACIÓN FÍSICA,https://admision.unmsm.edu.pe/Website20262/A/0...
5,807096,"Aliaga Torres, Juan Pablo Gabriel",EDUCACIÓN FÍSICA,1078.150,23.0,ALCANZÓ VACANTE,EDUCACIÓN FÍSICA,https://admision.unmsm.edu.pe/Website20262/A/0...
6,780965,"Alva Pecca, Ronel Franco",EDUCACIÓN FÍSICA,1072.225,27.0,ALCANZÓ VACANTE,EDUCACIÓN FÍSICA,https://admision.unmsm.edu.pe/Website20262/A/0...
7,716990,"Alvarez Dongo, Jeampiers",EDUCACIÓN FÍSICA,832.634,NaN,SIN OBSERVACIÓN,EDUCACIÓN FÍSICA,https://admision.unmsm.edu.pe/Website20262/A/0...
8,686533,"Alvis Vargas, Pedro Fabiano",EDUCACIÓN FÍSICA,974.888,59.0,ALCANZÓ VACANTE,EDUCACIÓN FÍSICA,https://admision.unmsm.edu.pe/Website20262/A/0...
9,787629,"Andia Paqui, Ysmael Mateo",EDUCACIÓN FÍSICA,896.496,94.0,ALCANZÓ VACANTE,EDUCACIÓN FÍSICA,https://admision.unmsm.edu.pe/Website20262/A/0...


In [25]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_edfisica.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: EDUCACIÓN FÍSICA
Vacantes obtenidas: 95
Puntaje máximo: 1248.171
Puntaje mínimo: 896.262


,codigo,apellidos_nombres,puntaje,merito_ep
0,814284,"Ramírez Ramos, Jostin Slash",1248.171,1.0
1,708761,"Medrano Berrios, Aaron Javier",1239.250,2.0
2,723675,"Gonzales Acosta, Joaquin Albhieri",1172.226,3.0
3,694303,"Bermudo Pérez, Jairo Josafat",1152.763,4.0
4,788667,"Santiago Santiago, Viviana Luz",1152.188,5.0
...,...,...,...,...
90,728955,"Cieza Quintana, Kati Paola",905.875,91.0
91,711140,"Montero Cueva, Vieri Giovanny",904.237,92.0
92,729146,"Rivas Ccoicca, Omar Dante",900.292,93.0
93,787629,"Andia Paqui, Ysmael Mateo",896.496,94.0


In [26]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/0611/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_edinicial.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/0611/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 179 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_edinicial.csv
   179 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          159
ALCANZÓ VACANTE           18
INHABILITADO (Art. 5)      2

── Estadísticas de puntaje ──
count     179.000
mean      727.158
std       148.385
min       362.500
25%       627.938
50%       713.750
75%       832.688
max      1108.125


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,751643,"Achulli Valeriano, Flor Milagros",EDUCACIÓN INICIAL,668.375,NaN,SIN OBSERVACIÓN,EDUCACIÓN INICIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
1,747979,"Acosta Herrera, Patricia Alexandra",EDUCACIÓN INICIAL,798.375,NaN,SIN OBSERVACIÓN,EDUCACIÓN INICIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
2,797640,"Alcazar Lopez, Ximena",EDUCACIÓN INICIAL,924.875,18.0,ALCANZÓ VACANTE,EDUCACIÓN INICIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
3,828804,"Aldunate Cabanillas, Jimena Qoricollor",EDUCACIÓN INICIAL,689.875,NaN,SIN OBSERVACIÓN,EDUCACIÓN INICIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
4,818591,"Anaya Perez, Ayelen Xiomara",EDUCACIÓN INICIAL,507.500,NaN,SIN OBSERVACIÓN,EDUCACIÓN INICIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
5,695732,"Andia Quispe, Jennevike Massiel",EDUCACIÓN INICIAL,746.750,NaN,SIN OBSERVACIÓN,EDUCACIÓN INICIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
6,840744,"Antaurco Aragón, Elizabeth Salomé",EDUCACIÓN INICIAL,799.625,NaN,SIN OBSERVACIÓN,EDUCACIÓN INICIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
7,840547,"Antunez Alvarez, Alison Regina",EDUCACIÓN INICIAL,889.875,NaN,SIN OBSERVACIÓN,EDUCACIÓN INICIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
8,694716,"Aponte Quispe, Melissa Emily",EDUCACIÓN INICIAL,683.000,NaN,SIN OBSERVACIÓN,EDUCACIÓN INICIAL,https://admision.unmsm.edu.pe/Website20262/A/0...
9,788517,"Ardiles Zavaleta, Maricielo",EDUCACIÓN INICIAL,562.250,NaN,SIN OBSERVACIÓN,EDUCACIÓN INICIAL,https://admision.unmsm.edu.pe/Website20262/A/0...


In [27]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_edinicial.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: EDUCACIÓN INICIAL
Vacantes obtenidas: 18
Puntaje máximo: 1108.125
Puntaje mínimo: 924.875


,codigo,apellidos_nombres,puntaje,merito_ep
0,703991,"Malasquez Llacsahuanga, Andrea Cosette",1108.125,1.0
1,690262,"Huayanay Adán, Medaly Estefany",1091.000,2.0
2,822218,"Pizarro Cabrera, Rubi Kimberly",1041.375,3.0
3,753605,"Romero Arce, Selene Maria",1034.375,4.0
4,687155,"Portilla Inca, Massyell Cielo Yuri",1028.000,5.0
5,693010,"Mantari Ochante, Gianella Milagros",1020.250,6.0
6,760311,"Castillo Carrero, Jhoana Lian",987.125,7.0
7,686673,"Rivera Centurión, Rihana Nahomy",978.000,8.0
8,844772,"Panduro Cardenas, Brisa Alejandra",975.500,9.0
9,694203,"Luque Felix, Karla Sofía",974.000,10.0


In [28]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/0612/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_edprimaria.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/0612/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 146 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_edprimaria.csv
   146 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN    126
ALCANZÓ VACANTE     20

── Estadísticas de puntaje ──
count     146.000
mean      792.002
std       166.403
min       384.625
25%       680.094
50%       788.625
75%       902.938
max      1267.375


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,741106,"Acco Ludena, Stefano Gadiel",EDUCACIÓN PRIMARIA,759.250,NaN,SIN OBSERVACIÓN,EDUCACIÓN PRIMARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
1,740655,"Acho Sanchez, Naydely Alexia",EDUCACIÓN PRIMARIA,746.750,NaN,SIN OBSERVACIÓN,EDUCACIÓN PRIMARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
2,724537,"Alvarez Gomez, Camila Maricel",EDUCACIÓN PRIMARIA,787.250,NaN,SIN OBSERVACIÓN,EDUCACIÓN PRIMARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
3,726843,"Antaurco Sucasaire, Kiara Grecia",EDUCACIÓN PRIMARIA,782.000,NaN,SIN OBSERVACIÓN,EDUCACIÓN PRIMARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
4,749535,"Anyosa Tumpay, Aracely Raquel",EDUCACIÓN PRIMARIA,951.625,NaN,SIN OBSERVACIÓN,EDUCACIÓN PRIMARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
5,690021,"Araujo Tito, Jannira Anahí",EDUCACIÓN PRIMARIA,740.125,NaN,SIN OBSERVACIÓN,EDUCACIÓN PRIMARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
6,755862,"Arroyo Martinez, Esmeralda Sofia",EDUCACIÓN PRIMARIA,704.875,NaN,SIN OBSERVACIÓN,EDUCACIÓN PRIMARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
7,801004,"Asencio Ochoa, Nicoll Cristina",EDUCACIÓN PRIMARIA,877.500,NaN,SIN OBSERVACIÓN,EDUCACIÓN PRIMARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
8,690583,"Atalaya Valentino, Betzabe Min",EDUCACIÓN PRIMARIA,1267.375,1.0,ALCANZÓ VACANTE,EDUCACIÓN PRIMARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
9,830252,"Atencia Caja, Milagros Nicole",EDUCACIÓN PRIMARIA,687.375,NaN,SIN OBSERVACIÓN,EDUCACIÓN PRIMARIA,https://admision.unmsm.edu.pe/Website20262/A/0...


In [29]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_edprimaria.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: EDUCACIÓN PRIMARIA
Vacantes obtenidas: 20
Puntaje máximo: 1267.375
Puntaje mínimo: 981.75


,codigo,apellidos_nombres,puntaje,merito_ep
0,690583,"Atalaya Valentino, Betzabe Min",1267.375,1.0
1,795076,"Machuca Reymundo, Mayori Betsabe",1137.750,2.0
2,720710,"Salluca Huahualuque, Mitzu Alisson",1137.125,3.0
3,723678,"Castro Garcia, Gady Milissa",1093.250,4.0
4,722811,"Laura Laura, Mía Qanka",1087.375,5.0
5,826044,"Lloclla Salazar, Adriana Carolina",1076.500,6.0
6,687244,"Olivera Auccatinco, Nicolle Angely",1060.750,7.0
7,795570,"Calle Ccoa, Francesca",1054.500,8.0
8,756178,"Vales Misajel, Jany Mercedes Karolina",1047.625,9.0
9,753608,"Villa Portilla, Maria Milagros",1044.750,10.0


In [30]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/0613/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_edsecundaria.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/0613/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 361 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_edsecundaria.csv
   361 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN    282
ALCANZÓ VACANTE     76
AUSENTE              3

── Estadísticas de puntaje ──
count     358.000
mean      830.826
std       172.308
min       375.625
25%       721.188
50%       833.188
75%       949.500
max      1371.375


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,732432,"Abrego Bonifacio, Brisa Anyeli",EDUCACIÓN SECUNDARIA,809.000,NaN,SIN OBSERVACIÓN,EDUCACIÓN SECUNDARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
1,842324,"Acuña Gamarra, Isabel Alexandra",EDUCACIÓN SECUNDARIA,695.500,NaN,SIN OBSERVACIÓN,EDUCACIÓN SECUNDARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
2,832751,"Adriano Valencia, Luz Bella",EDUCACIÓN SECUNDARIA,894.625,NaN,SIN OBSERVACIÓN,EDUCACIÓN SECUNDARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
3,829820,"Aguirre Mendoza, Brilly Carolina",EDUCACIÓN SECUNDARIA,979.125,68.0,ALCANZÓ VACANTE,EDUCACIÓN SECUNDARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
4,768731,"Aguirre Monzón, Maria Isabel Yamileth",EDUCACIÓN SECUNDARIA,753.500,NaN,SIN OBSERVACIÓN,EDUCACIÓN SECUNDARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
5,800856,"Aguirre Rivera, Valery Arlette",EDUCACIÓN SECUNDARIA,763.500,NaN,SIN OBSERVACIÓN,EDUCACIÓN SECUNDARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
6,717716,"Aguirre Wall, Franshesco Dominic",EDUCACIÓN SECUNDARIA,832.000,NaN,SIN OBSERVACIÓN,EDUCACIÓN SECUNDARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
7,691149,"Alania Vargas, Trayce Nicol",EDUCACIÓN SECUNDARIA,580.250,NaN,SIN OBSERVACIÓN,EDUCACIÓN SECUNDARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
8,745653,"Alarcon Baez, Ronald",EDUCACIÓN SECUNDARIA,1021.375,47.0,ALCANZÓ VACANTE,EDUCACIÓN SECUNDARIA,https://admision.unmsm.edu.pe/Website20262/A/0...
9,800890,"Aldazabal Cruz, Noelia Sayuri",EDUCACIÓN SECUNDARIA,506.375,NaN,SIN OBSERVACIÓN,EDUCACIÓN SECUNDARIA,https://admision.unmsm.edu.pe/Website20262/A/0...


In [31]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_edsecundaria.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: EDUCACIÓN SECUNDARIA
Vacantes obtenidas: 76
Puntaje máximo: 1371.375
Puntaje mínimo: 970.0


,codigo,apellidos_nombres,puntaje,merito_ep
0,752097,"Alva Alba, Nain Gabriel",1371.375,1.0
1,715969,"Valverde Gonzales, Gabriela Medali",1274.875,2.0
2,741473,"Coronado Comitivos, Alvaro Leonel",1252.625,3.0
3,734211,"Tejada Avellaneda, Darien Antoine",1215.500,4.0
4,777200,"Ccotohuanca Patricio, José Luis Arturo",1213.125,5.0
...,...,...,...,...
71,735738,"Tapullima Mendez, Daniela Anelins",977.750,72.0
72,750960,"Santiago Flores, Geremi",975.125,73.0
73,820996,"Ballon Layme, Victor Hugo",974.875,74.0
74,703652,"Arbildo Espino, Stefano Santos",970.000,76.0


In [32]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/142/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_estadistica.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/142/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 65 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_estadistica.csv
   65 filas  ×  8 columnas

── Observaciones ──
observacion
ALCANZÓ VACANTE          63
INHABILITADO (Art. 5)     2

── Estadísticas de puntaje ──
count      65.000
mean      839.877
std       188.950
min       396.750
25%       713.000
50%       851.250
75%       931.125
max      1446.750


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,212770,"Aguilar Mogollon, Yerson Aarón",ESTADÍSTICA,798.000,43.0,ALCANZÓ VACANTE,ESTADÍSTICA,https://admision.unmsm.edu.pe/Website20262/A/1...
1,211868,"Alomia Fernandez, Jose Luis Jr.",ESTADÍSTICA,927.750,19.0,ALCANZÓ VACANTE,ESTADÍSTICA,https://admision.unmsm.edu.pe/Website20262/A/1...
2,215963,"Aniceto Mamani, Jefferson Aaron",ESTADÍSTICA,564.500,61.0,ALCANZÓ VACANTE,ESTADÍSTICA,https://admision.unmsm.edu.pe/Website20262/A/1...
3,215255,"Arias Sapa, Augusto Andre",ESTADÍSTICA,911.750,21.0,ALCANZÓ VACANTE,ESTADÍSTICA,https://admision.unmsm.edu.pe/Website20262/A/1...
4,215990,"Arteaga Machuca, Hugo Lucio",ESTADÍSTICA,830.625,36.0,ALCANZÓ VACANTE,ESTADÍSTICA,https://admision.unmsm.edu.pe/Website20262/A/1...
5,214027,"Ascue Rondon, Claudia Alexandra",ESTADÍSTICA,396.750,65.0,ALCANZÓ VACANTE,ESTADÍSTICA,https://admision.unmsm.edu.pe/Website20262/A/1...
6,218236,"Basurto Chavez, Ana Karina",ESTADÍSTICA,877.375,30.0,ALCANZÓ VACANTE,ESTADÍSTICA,https://admision.unmsm.edu.pe/Website20262/A/1...
7,216685,"Bernardo Jurado, Piero Teo",ESTADÍSTICA,931.125,17.0,ALCANZÓ VACANTE,ESTADÍSTICA,https://admision.unmsm.edu.pe/Website20262/A/1...
8,214047,"Cabrera Ore, Angello Vincent",ESTADÍSTICA,660.625,55.0,ALCANZÓ VACANTE,ESTADÍSTICA,https://admision.unmsm.edu.pe/Website20262/A/1...
9,216703,"Chua Sayaverde, Valeria Amelia",ESTADÍSTICA,894.375,26.0,ALCANZÓ VACANTE,ESTADÍSTICA,https://admision.unmsm.edu.pe/Website20262/A/1...


In [33]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_estadistica.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: ESTADÍSTICA
Vacantes obtenidas: 63
Puntaje máximo: 1446.75
Puntaje mínimo: 396.75


,codigo,apellidos_nombres,puntaje,merito_ep
0,218653,"Guevara Peña, Alexis Adrian",1446.750,1.0
1,215827,"Ramos Huacho, Aleli Ana",1314.250,2.0
2,213449,"Leon Alanoca, Valentina Guisell",1210.375,3.0
3,218256,"Gonzales Palacios, Juan José Manuel",1209.125,4.0
4,212836,"Quincho Esteban, Mark Blary",1066.500,5.0
...,...,...,...,...
58,215963,"Aniceto Mamani, Jefferson Aaron",564.500,61.0
59,211715,"Landeo Clerque, Thiago Alejandro",554.750,62.0
60,210737,"Maguiña Huachara, Alexsander Guisseppe",516.625,63.0
61,211896,"Soto Ipanaqué, Anderson Iván",502.875,64.0


In [1]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/204/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_ia.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/204/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 512 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_ia.csv
   512 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          457
ALCANZÓ VACANTE           44
AUSENTE                    6
INHABILITADO (Art. 5)      5

── Estadísticas de puntaje ──
count     506.000
mean      879.454
std       179.613
min       345.000
25%       743.406
50%       878.688
75%      1006.500
max      1465.000


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,373848,"Abanto Ccahuana, Nicole Alexandra",INGENIERÍA DE INTELIGENCIA ARTIFICIAL,831.000,NaN,SIN OBSERVACIÓN,INGENIERÍA DE INTELIGENCIA ARTIFICIAL,https://admision.unmsm.edu.pe/Website20262/A/2...
1,341380,"Abanto Ramirez, Abraham Leonardo",INGENIERÍA DE INTELIGENCIA ARTIFICIAL,969.875,NaN,SIN OBSERVACIÓN,INGENIERÍA DE INTELIGENCIA ARTIFICIAL,https://admision.unmsm.edu.pe/Website20262/A/2...
2,383633,"Accate Marallano, Steven Alonso",INGENIERÍA DE INTELIGENCIA ARTIFICIAL,965.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE INTELIGENCIA ARTIFICIAL,https://admision.unmsm.edu.pe/Website20262/A/2...
3,382198,"Achulli Rivera, Kiara Camila",INGENIERÍA DE INTELIGENCIA ARTIFICIAL,1284.000,7.0,ALCANZÓ VACANTE,INGENIERÍA DE INTELIGENCIA ARTIFICIAL,https://admision.unmsm.edu.pe/Website20262/A/2...
4,354738,"Aguayo Damacen, Jhoan Galois",INGENIERÍA DE INTELIGENCIA ARTIFICIAL,1030.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE INTELIGENCIA ARTIFICIAL,https://admision.unmsm.edu.pe/Website20262/A/2...
5,358935,"Aguirre Huaringa, Jhoseph Xavier",INGENIERÍA DE INTELIGENCIA ARTIFICIAL,1201.125,21.0,ALCANZÓ VACANTE,INGENIERÍA DE INTELIGENCIA ARTIFICIAL,https://admision.unmsm.edu.pe/Website20262/A/2...
6,313580,"Albornoz Ortiz, Keyla",INGENIERÍA DE INTELIGENCIA ARTIFICIAL,588.500,NaN,SIN OBSERVACIÓN,INGENIERÍA DE INTELIGENCIA ARTIFICIAL,https://admision.unmsm.edu.pe/Website20262/A/2...
7,344268,"Alcantara Melgarejo, Erick",INGENIERÍA DE INTELIGENCIA ARTIFICIAL,839.250,NaN,SIN OBSERVACIÓN,INGENIERÍA DE INTELIGENCIA ARTIFICIAL,https://admision.unmsm.edu.pe/Website20262/A/2...
8,387067,"Alcántara Murga, Diego Jhair",INGENIERÍA DE INTELIGENCIA ARTIFICIAL,1016.250,NaN,SIN OBSERVACIÓN,INGENIERÍA DE INTELIGENCIA ARTIFICIAL,https://admision.unmsm.edu.pe/Website20262/A/2...
9,326441,"Alcarraz Gonzales, Luis Miguel",INGENIERÍA DE INTELIGENCIA ARTIFICIAL,902.625,NaN,INHABILITADO (Art. 5),INGENIERÍA DE INTELIGENCIA ARTIFICIAL,https://admision.unmsm.edu.pe/Website20262/A/2...


In [2]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_ia.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERÍA DE INTELIGENCIA ARTIFICIAL
Vacantes obtenidas: 44
Puntaje máximo: 1465.0
Puntaje mínimo: 1131.0


,codigo,apellidos_nombres,puntaje,merito_ep
0,388113,"Inga Cancho, Dilan Sebastian",1465.000,1.0
1,367558,"Castro Neri, David Santiago",1359.375,2.0
2,344878,"Collantes Atalaya, Gino Sebastian",1339.375,3.0
3,319780,"Melo Ramos, Patrick Jusephy",1305.125,4.0
4,387956,"Cardenas Palma, Santiago De Jesus",1301.000,5.0
5,393599,"Lopez Bonifacio, Danny Junior",1292.000,6.0
6,382198,"Achulli Rivera, Kiara Camila",1284.000,7.0
7,354627,"Huaman Chapoñan, Cristian Bryan",1277.125,8.0
8,317298,"Morales Alejandro, Anderson Jose",1273.750,9.0
9,354047,"Avila Becerra, Arturo Rodrigo",1269.750,10.0


In [3]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/167/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_civil.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/167/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 642 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_civil.csv
   642 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          587
ALCANZÓ VACANTE           37
INHABILITADO (Art. 5)     10
AUSENTE                    8

── Estadísticas de puntaje ──
count     634.000
mean      910.017
std       215.777
min       412.125
25%       750.656
50%       895.188
75%      1051.750
max      1521.500


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,347086,"Abarca Castillo, Fabian Yony",INGENIERÍA CIVIL,1350.250,17.0,ALCANZÓ VACANTE,INGENIERÍA CIVIL,https://admision.unmsm.edu.pe/Website20262/A/1...
1,317687,"Abollaneda Mamani, Patrick Yhojan",INGENIERÍA CIVIL,1254.875,NaN,SIN OBSERVACIÓN,INGENIERÍA CIVIL,https://admision.unmsm.edu.pe/Website20262/A/1...
2,373379,"Acevedo Herrera, Meilhyn Claris",INGENIERÍA CIVIL,1122.375,NaN,SIN OBSERVACIÓN,INGENIERÍA CIVIL,https://admision.unmsm.edu.pe/Website20262/A/1...
3,310073,"Acha Rojas, Joan Francisco",INGENIERÍA CIVIL,852.125,NaN,SIN OBSERVACIÓN,INGENIERÍA CIVIL,https://admision.unmsm.edu.pe/Website20262/A/1...
4,376815,"Acosta Quesnay, Michael Del Piero",INGENIERÍA CIVIL,1007.625,NaN,SIN OBSERVACIÓN,INGENIERÍA CIVIL,https://admision.unmsm.edu.pe/Website20262/A/1...
5,372965,"Acuña Becerra, Piero Alessandro",INGENIERÍA CIVIL,737.625,NaN,SIN OBSERVACIÓN,INGENIERÍA CIVIL,https://admision.unmsm.edu.pe/Website20262/A/1...
6,325491,"Acuña Mallqui, Max Gabriel",INGENIERÍA CIVIL,775.875,NaN,SIN OBSERVACIÓN,INGENIERÍA CIVIL,https://admision.unmsm.edu.pe/Website20262/A/1...
7,356205,"Aguilar Mejia, Jose Julian",INGENIERÍA CIVIL,864.375,NaN,SIN OBSERVACIÓN,INGENIERÍA CIVIL,https://admision.unmsm.edu.pe/Website20262/A/1...
8,353553,"Aguilar Morocho, Sebastian Joseph",INGENIERÍA CIVIL,573.875,NaN,SIN OBSERVACIÓN,INGENIERÍA CIVIL,https://admision.unmsm.edu.pe/Website20262/A/1...
9,326780,"Alanya Tito, Ever Anthony",INGENIERÍA CIVIL,624.000,NaN,SIN OBSERVACIÓN,INGENIERÍA CIVIL,https://admision.unmsm.edu.pe/Website20262/A/1...


In [4]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_civil.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERÍA CIVIL
Vacantes obtenidas: 37
Puntaje máximo: 1521.5
Puntaje mínimo: 1269.75


,codigo,apellidos_nombres,puntaje,merito_ep
0,329798,"Pumahuallca Horna, Miguel Mathías",1521.500,1.0
1,365995,"Sanchez Uscamayta, Axel Vagner",1471.750,2.0
2,379870,"Carlos Suarez, Jean Piere",1465.000,3.0
3,351380,"Quispe Velasquez, Luis Delfin",1465.000,4.0
4,334524,"Casiano Wilson, Moises Jesus",1440.750,5.0
5,323088,"Burgos Vidarte, José Fernando",1430.750,6.0
6,344335,"Sanchez Vasquez, Jhimy Harold",1422.750,7.0
7,351750,"Tullume Belen, Max Arthur",1421.625,8.0
8,394259,"Alfaro Quispe, Fernando José",1418.750,9.0
9,348497,"Ayvar Nazario, Matias Aaron",1400.500,10.0


In [5]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/173/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_seguridad.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/173/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 79 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_seguridad.csv
   79 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          48
ALCANZÓ VACANTE          30
INHABILITADO (Art. 5)     1

── Estadísticas de puntaje ──
count      79.000
mean      860.239
std       154.170
min       521.250
25%       752.375
50%       846.375
75%       956.000
max      1252.625


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,334555,"Achic Vasquez, Estrella Yamil",INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,970.000,18.0,ALCANZÓ VACANTE,INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,https://admision.unmsm.edu.pe/Website20262/A/1...
1,384607,"Achulli Truevas, Judith Yomali",INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,835.250,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,https://admision.unmsm.edu.pe/Website20262/A/1...
2,382872,"Aguirre Navarro, Jordan",INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,1071.625,7.0,ALCANZÓ VACANTE,INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,https://admision.unmsm.edu.pe/Website20262/A/1...
3,383383,"Aguirre Rodriguez, Caleb Jonas",INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,897.500,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,https://admision.unmsm.edu.pe/Website20262/A/1...
4,344800,"Alabrin Soto, Vanessa Aracely",INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,660.000,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,https://admision.unmsm.edu.pe/Website20262/A/1...
5,361628,"Alegría Quispe, Isis Alessandra",INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,823.250,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,https://admision.unmsm.edu.pe/Website20262/A/1...
6,340347,"Alvaro Espinoza, Karen Milagros",INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,759.000,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,https://admision.unmsm.edu.pe/Website20262/A/1...
7,395255,"Arias Guzman, Sebastian Daniel",INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,766.500,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,https://admision.unmsm.edu.pe/Website20262/A/1...
8,323004,"Barja Palma, Cristopher",INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,707.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,https://admision.unmsm.edu.pe/Website20262/A/1...
9,315790,"Caceres Gonzales, Liumbert David",INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,684.000,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO,https://admision.unmsm.edu.pe/Website20262/A/1...


In [6]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_seguridad.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERÍA DE SEGURIDAD Y SALUD EN EL TRABAJO
Vacantes obtenidas: 30
Puntaje máximo: 1252.625
Puntaje mínimo: 902.625


,codigo,apellidos_nombres,puntaje,merito_ep
0,335150,"Gonzales Paucar, Sebastian Nícolas",1252.625,1.0
1,397068,"Laurente Aviles, Jhon Wily",1228.625,2.0
2,379484,"Dextre Juarez, Adrian Francesco",1143.500,3.0
3,312273,"Velasquez Apaza, Cristian Piero",1142.750,4.0
4,352381,"Moreno Cahuaza, Andher Gadhit",1123.500,5.0
5,377385,"Lopez Chuchon, Betzabeth Selena",1084.125,6.0
6,382872,"Aguirre Navarro, Jordan",1071.625,7.0
7,365176,"Montes Barrientos, Jesus Simon",1062.500,8.0
8,360041,"Capellino Godoy, Aaron Eduardo",1058.500,9.0
9,368968,"Talledo Garcia, Rodrigo Nicolas",1048.125,10.0


In [7]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/201/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_sistemas.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/201/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 875 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_sistemas.csv
   875 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          805
ALCANZÓ VACANTE           50
AUSENTE                   13
INHABILITADO (Art. 5)      7

── Estadísticas de puntaje ──
count     862.000
mean      878.758
std       225.099
min       254.750
25%       712.375
50%       858.250
75%      1020.250
max      1557.500


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,367338,"Abanto Ramos, D'Alessandro",INGENIERÍA DE SISTEMAS,789.000,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SISTEMAS,https://admision.unmsm.edu.pe/Website20262/A/2...
1,350381,"Acero Quiñonez, Frank Deyvis",INGENIERÍA DE SISTEMAS,907.750,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SISTEMAS,https://admision.unmsm.edu.pe/Website20262/A/2...
2,374178,"Acevedo Llallahui, Jack Arlington",INGENIERÍA DE SISTEMAS,NaN,NaN,AUSENTE,INGENIERÍA DE SISTEMAS,https://admision.unmsm.edu.pe/Website20262/A/2...
3,388366,"Acosta Gordillo, David Rafael",INGENIERÍA DE SISTEMAS,855.125,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SISTEMAS,https://admision.unmsm.edu.pe/Website20262/A/2...
4,323230,"Acosta Rocca, Deyvis Nicolas",INGENIERÍA DE SISTEMAS,742.750,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SISTEMAS,https://admision.unmsm.edu.pe/Website20262/A/2...
5,334937,"Acosta Vilca, Fatima Valeria",INGENIERÍA DE SISTEMAS,711.875,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SISTEMAS,https://admision.unmsm.edu.pe/Website20262/A/2...
6,395453,"Adrianzén Ramos, Ibrahim Mathias Marto",INGENIERÍA DE SISTEMAS,1017.250,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SISTEMAS,https://admision.unmsm.edu.pe/Website20262/A/2...
7,333169,"Aguilar Briones, Diego Omar",INGENIERÍA DE SISTEMAS,1009.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SISTEMAS,https://admision.unmsm.edu.pe/Website20262/A/2...
8,344640,"Aguirre Arias, Jesus Arnaldo",INGENIERÍA DE SISTEMAS,798.500,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SISTEMAS,https://admision.unmsm.edu.pe/Website20262/A/2...
9,312329,"Aguirre Macedo, Dante Leonidas",INGENIERÍA DE SISTEMAS,733.625,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SISTEMAS,https://admision.unmsm.edu.pe/Website20262/A/2...


In [8]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_sistemas.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERÍA DE SISTEMAS
Vacantes obtenidas: 50
Puntaje máximo: 1557.5
Puntaje mínimo: 1268.0


,codigo,apellidos_nombres,puntaje,merito_ep
0,316398,"Castro Suclupe, Kevin David",1557.500,1.0
1,323395,"Garcia Rubio, Jose Luis",1515.250,2.0
2,318077,"Serna Villon, Nincoll Oswaldo",1503.250,3.0
3,337010,"Magallanes Cordova, Luis Alberto",1502.125,4.0
4,384547,"Roque Tuñoque, Stefany Eylith",1478.125,5.0
5,338350,"Coaquira Orihuela, Dayana Ruth",1472.375,6.0
6,326867,"Ortiz Mendiburu, Mathias Gabriel",1466.625,7.0
7,377862,"Arque Vasquez, Jose Estefano",1451.750,8.0
8,360589,"Ramirez Ramos, Juan Alejandro",1446.750,9.0
9,312117,"Calderón Rojas, Saih Ramiro",1434.750,10.0


In [9]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/193/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_teleco.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/193/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 125 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_teleco.csv
   125 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN    81
ALCANZÓ VACANTE    44

── Estadísticas de puntaje ──
count     125.000
mean      946.820
std       186.999
min       510.375
25%       826.125
50%       956.750
75%      1062.500
max      1410.750


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,377549,"Abanto Torres, Renato Fadull",INGENIERÍA DE TELECOMUNICACIONES,1115.500,22.0,ALCANZÓ VACANTE,INGENIERÍA DE TELECOMUNICACIONES,https://admision.unmsm.edu.pe/Website20262/A/1...
1,338068,"Adan Cabia, Frank David",INGENIERÍA DE TELECOMUNICACIONES,766.500,NaN,SIN OBSERVACIÓN,INGENIERÍA DE TELECOMUNICACIONES,https://admision.unmsm.edu.pe/Website20262/A/1...
2,360385,"Agama Quiliche, Fernando William",INGENIERÍA DE TELECOMUNICACIONES,644.000,NaN,SIN OBSERVACIÓN,INGENIERÍA DE TELECOMUNICACIONES,https://admision.unmsm.edu.pe/Website20262/A/1...
3,394785,"Alarcon Espinoza, Stefano Russell",INGENIERÍA DE TELECOMUNICACIONES,1053.750,36.0,ALCANZÓ VACANTE,INGENIERÍA DE TELECOMUNICACIONES,https://admision.unmsm.edu.pe/Website20262/A/1...
4,386070,"Alcantara Meza, Armando Josue",INGENIERÍA DE TELECOMUNICACIONES,1047.875,38.0,ALCANZÓ VACANTE,INGENIERÍA DE TELECOMUNICACIONES,https://admision.unmsm.edu.pe/Website20262/A/1...
5,383884,"Allccahuaman Llampi, Daniel Anthony",INGENIERÍA DE TELECOMUNICACIONES,914.625,NaN,SIN OBSERVACIÓN,INGENIERÍA DE TELECOMUNICACIONES,https://admision.unmsm.edu.pe/Website20262/A/1...
6,343350,"Alviz Lopez, Jair Alonso",INGENIERÍA DE TELECOMUNICACIONES,966.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE TELECOMUNICACIONES,https://admision.unmsm.edu.pe/Website20262/A/1...
7,339424,"Alzamora Quiñones, Alejandro Isaias",INGENIERÍA DE TELECOMUNICACIONES,1084.750,28.0,ALCANZÓ VACANTE,INGENIERÍA DE TELECOMUNICACIONES,https://admision.unmsm.edu.pe/Website20262/A/1...
8,343691,"Amambal Huaman, Diego Madinson",INGENIERÍA DE TELECOMUNICACIONES,1213.125,10.0,ALCANZÓ VACANTE,INGENIERÍA DE TELECOMUNICACIONES,https://admision.unmsm.edu.pe/Website20262/A/1...
9,375415,"Anaya Venegas, Anderson Neysser",INGENIERÍA DE TELECOMUNICACIONES,1410.750,1.0,ALCANZÓ VACANTE,INGENIERÍA DE TELECOMUNICACIONES,https://admision.unmsm.edu.pe/Website20262/A/1...


In [10]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_teleco.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERÍA DE TELECOMUNICACIONES
Vacantes obtenidas: 44
Puntaje máximo: 1410.75
Puntaje mínimo: 1025.375


,codigo,apellidos_nombres,puntaje,merito_ep
0,375415,"Anaya Venegas, Anderson Neysser",1410.750,1.0
1,373241,"Torres Bautista, Eder Juan Sebasthian",1351.375,2.0
2,390407,"Zuñiga Macchiavello, Mateo Rodrigo",1331.375,3.0
3,344991,"Antunez Ysturiz, Wilder Daniel",1300.500,4.0
4,376032,"Gutierrez Isidro, Fernando Alonso",1290.875,5.0
5,398659,"Estrada Zuasnabar, Jose Rodolfo",1282.875,6.0
6,358204,"Roca Ccacca, William Oscar",1252.625,7.0
7,346072,"Cahuana Canaza, Aziel Mathieu",1249.750,8.0
8,310373,"Quiñonez Mozo, Brayan Sebastian",1238.875,9.0
9,343691,"Amambal Huaman, Diego Madinson",1213.125,10.0


In [11]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/175/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_transportes.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/175/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 172 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_transportes.csv
   172 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN    128
ALCANZÓ VACANTE     44

── Estadísticas de puntaje ──
count     172.000
mean      815.821
std       162.479
min       449.250
25%       703.562
50%       812.625
75%       913.094
max      1275.375


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,329025,"Abanto Mendoza, Laura Anais",INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,930.875,40.0,ALCANZÓ VACANTE,INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,https://admision.unmsm.edu.pe/Website20262/A/1...
1,325948,"Aguilar Villagomez, Edgar Junior",INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,1025.250,19.0,ALCANZÓ VACANTE,INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,https://admision.unmsm.edu.pe/Website20262/A/1...
2,393058,"Aguirre Perfecto, Samuel Elias",INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,965.250,33.0,ALCANZÓ VACANTE,INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,https://admision.unmsm.edu.pe/Website20262/A/1...
3,343172,"Alminagorda Flores, Noe Eleazar",INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,893.500,NaN,SIN OBSERVACIÓN,INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,https://admision.unmsm.edu.pe/Website20262/A/1...
4,317707,"Altamirano Julca, Jefferson",INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,885.500,NaN,SIN OBSERVACIÓN,INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,https://admision.unmsm.edu.pe/Website20262/A/1...
5,374327,"Alvarado Cabrera, Piero Valentin",INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,1063.625,11.0,ALCANZÓ VACANTE,INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,https://admision.unmsm.edu.pe/Website20262/A/1...
6,352212,"Alvarado Galarza, Katherine Silvia",INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,779.875,NaN,SIN OBSERVACIÓN,INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,https://admision.unmsm.edu.pe/Website20262/A/1...
7,360325,"Alvarado Huayllani, Oscar Andre",INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,839.250,NaN,SIN OBSERVACIÓN,INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,https://admision.unmsm.edu.pe/Website20262/A/1...
8,333766,"Alvarez Sifuentes, Jesus Cluber",INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,616.750,NaN,SIN OBSERVACIÓN,INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,https://admision.unmsm.edu.pe/Website20262/A/1...
9,382875,"Alvarez Taipe, Jhon Linder",INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,813.000,NaN,SIN OBSERVACIÓN,INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS,https://admision.unmsm.edu.pe/Website20262/A/1...


In [12]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_transportes.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERIA DE TRANSPORTES Y SISTEMAS FERROVIARIOS
Vacantes obtenidas: 44
Puntaje máximo: 1275.375
Puntaje mínimo: 912.875


,codigo,apellidos_nombres,puntaje,merito_ep
0,386014,"Barzola Del Carpio, Alessandro Piero",1275.375,1.0
1,381352,"Jara Sanchez, Osmar Santiago",1240.625,2.0
2,333619,"Chaccha Quispe, Thiago Mateo",1185.250,3.0
3,348194,"Perez Quispe, Henry Jostin",1156.125,4.0
4,394476,"Armuto Sabrera, Nicole Sthefany",1143.000,5.0
5,350875,"Vasquez Mori, Andres",1125.875,6.0
6,338604,"Piscoya Bustamante, Juan Manuel",1088.125,7.0
7,368365,"Vicerrel Chávez, José Alberto",1079.625,8.0
8,365432,"Pagan Luis, Gonzalo Emanuel",1077.750,9.0
9,376926,"Deudor Astete, Alizee Avril",1068.750,10.0


In [13]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/191/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_electronica.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/191/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 255 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_electronica.csv
   255 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          191
ALCANZÓ VACANTE           61
AUSENTE                    2
INHABILITADO (Art. 5)      1

── Estadísticas de puntaje ──
count     253.000
mean      916.825
std       198.629
min       362.000
25%       786.750
50%       911.125
75%      1058.500
max      1407.250


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,359513,"Aburto Bonilla, Josue Marco",INGENIERÍA ELECTRÓNICA,1075.875,56.0,ALCANZÓ VACANTE,INGENIERÍA ELECTRÓNICA,https://admision.unmsm.edu.pe/Website20262/A/1...
1,378275,"Agama De La Cruz, Leonel Angel",INGENIERÍA ELECTRÓNICA,1075.625,57.0,ALCANZÓ VACANTE,INGENIERÍA ELECTRÓNICA,https://admision.unmsm.edu.pe/Website20262/A/1...
2,344010,"Aguilar Quispe, Yandel Sthee",INGENIERÍA ELECTRÓNICA,1014.625,NaN,SIN OBSERVACIÓN,INGENIERÍA ELECTRÓNICA,https://admision.unmsm.edu.pe/Website20262/A/1...
3,345548,"Aguilar Ramirez, Kevin Antony",INGENIERÍA ELECTRÓNICA,1331.750,4.0,ALCANZÓ VACANTE,INGENIERÍA ELECTRÓNICA,https://admision.unmsm.edu.pe/Website20262/A/1...
4,373284,"Alarcón Céspedes, Jhanpool Alexander",INGENIERÍA ELECTRÓNICA,826.125,NaN,SIN OBSERVACIÓN,INGENIERÍA ELECTRÓNICA,https://admision.unmsm.edu.pe/Website20262/A/1...
5,367668,"Alcantara Delgado, Fabrcio Axel",INGENIERÍA ELECTRÓNICA,942.250,NaN,SIN OBSERVACIÓN,INGENIERÍA ELECTRÓNICA,https://admision.unmsm.edu.pe/Website20262/A/1...
6,346253,"Almandos Natividad, Yerimen Brissel",INGENIERÍA ELECTRÓNICA,948.875,NaN,SIN OBSERVACIÓN,INGENIERÍA ELECTRÓNICA,https://admision.unmsm.edu.pe/Website20262/A/1...
7,349372,"Alvarado Orellana, Maycol Esteban",INGENIERÍA ELECTRÓNICA,1388.500,2.0,ALCANZÓ VACANTE,INGENIERÍA ELECTRÓNICA,https://admision.unmsm.edu.pe/Website20262/A/1...
8,380753,"Alvarez Menendez, Alessa Dasha",INGENIERÍA ELECTRÓNICA,1040.375,NaN,SIN OBSERVACIÓN,INGENIERÍA ELECTRÓNICA,https://admision.unmsm.edu.pe/Website20262/A/1...
9,384446,"Alvarez Sandoval, Angel Manuel",INGENIERÍA ELECTRÓNICA,630.875,NaN,SIN OBSERVACIÓN,INGENIERÍA ELECTRÓNICA,https://admision.unmsm.edu.pe/Website20262/A/1...


In [14]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_electronica.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERÍA ELECTRÓNICA
Vacantes obtenidas: 61
Puntaje máximo: 1407.25
Puntaje mínimo: 1065.875


,codigo,apellidos_nombres,puntaje,merito_ep
0,363222,"Ruiz Paulino, Johan",1407.250,1.0
1,349372,"Alvarado Orellana, Maycol Esteban",1388.500,2.0
2,343161,"Montes Hoyos, Erick Franco",1343.375,3.0
3,345548,"Aguilar Ramirez, Kevin Antony",1331.750,4.0
4,373993,"Llacctas Diaz, Esteban Antonio",1298.750,5.0
...,...,...,...,...
56,378275,"Agama De La Cruz, Leonel Angel",1075.625,57.0
57,313637,"Miñano Castillo, Fabian Nicolas",1073.875,58.0
58,342327,"Herrera Osorio, Celeste Romina",1073.000,59.0
59,384804,"Caqui Chamorro, Keyla Katherine",1066.750,60.0


In [15]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/171/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_industrial.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/171/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 866 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_industrial.csv
   866 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          759
ALCANZÓ VACANTE           90
AUSENTE                    9
INHABILITADO (Art. 5)      8

── Estadísticas de puntaje ──
count     857.000
mean      894.344
std       214.012
min       404.500
25%       730.750
50%       884.250
75%      1046.500
max      1613.375


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,350005,"Abregu Licla, Rodrigo Alexis",INGENIERÍA INDUSTRIAL,1126.375,NaN,SIN OBSERVACIÓN,INGENIERÍA INDUSTRIAL,https://admision.unmsm.edu.pe/Website20262/A/1...
1,387375,"Abregu Ñahuinripa, Leonel Steffano",INGENIERÍA INDUSTRIAL,978.000,NaN,SIN OBSERVACIÓN,INGENIERÍA INDUSTRIAL,https://admision.unmsm.edu.pe/Website20262/A/1...
2,315675,"Acevedo Nunez, Cielo Nicole",INGENIERÍA INDUSTRIAL,682.250,NaN,SIN OBSERVACIÓN,INGENIERÍA INDUSTRIAL,https://admision.unmsm.edu.pe/Website20262/A/1...
3,354866,"Acosta Querevalu, Estrella Milagros",INGENIERÍA INDUSTRIAL,1012.250,NaN,SIN OBSERVACIÓN,INGENIERÍA INDUSTRIAL,https://admision.unmsm.edu.pe/Website20262/A/1...
4,329811,"Acuña Palacios, Valeria Lizbeth",INGENIERÍA INDUSTRIAL,729.750,NaN,SIN OBSERVACIÓN,INGENIERÍA INDUSTRIAL,https://admision.unmsm.edu.pe/Website20262/A/1...
5,378570,"Adriano Arribasplata, Alvaro Antonio",INGENIERÍA INDUSTRIAL,857.125,NaN,SIN OBSERVACIÓN,INGENIERÍA INDUSTRIAL,https://admision.unmsm.edu.pe/Website20262/A/1...
6,375960,"Aguila Gaspar, Juan Jose",INGENIERÍA INDUSTRIAL,717.625,NaN,SIN OBSERVACIÓN,INGENIERÍA INDUSTRIAL,https://admision.unmsm.edu.pe/Website20262/A/1...
7,389924,"Aguilar Bendita, Pamela",INGENIERÍA INDUSTRIAL,977.000,NaN,SIN OBSERVACIÓN,INGENIERÍA INDUSTRIAL,https://admision.unmsm.edu.pe/Website20262/A/1...
8,390632,"Aguilar Echevarria, Mathías Angel",INGENIERÍA INDUSTRIAL,1343.375,20.0,ALCANZÓ VACANTE,INGENIERÍA INDUSTRIAL,https://admision.unmsm.edu.pe/Website20262/A/1...
9,313318,"Aguilar Perales, Andres",INGENIERÍA INDUSTRIAL,996.250,NaN,SIN OBSERVACIÓN,INGENIERÍA INDUSTRIAL,https://admision.unmsm.edu.pe/Website20262/A/1...


In [16]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_industrial.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERÍA INDUSTRIAL
Vacantes obtenidas: 90
Puntaje máximo: 1613.375
Puntaje mínimo: 1164.125


,codigo,apellidos_nombres,puntaje,merito_ep
0,363732,"Bruno Anchaise, Giovanni Paolo",1613.375,1.0
1,359250,"Dextre Moran, Maria Fernanda",1506.000,2.0
2,371466,"Toribio Jara, Oscar Fernando",1467.875,3.0
3,394949,"Alderete Castillo, Fernando Alonso",1467.875,4.0
4,322242,"Vasquez Uribe, Alessandro José",1447.875,5.0
...,...,...,...,...
85,389996,"Cabezas Aspajo, Luciano Adrien",1168.125,85.0
86,391223,"Delgado Custodio, Anny Stephanie",1166.750,87.0
87,339114,"Castro Gamarra, Evanjhely Solange",1165.750,88.0
88,355253,"Bonifacio Quispe, Camila Alexandra",1165.250,89.0


In [17]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/195/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_mecatronica.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/195/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 291 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_mecatronica.csv
   291 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          243
ALCANZÓ VACANTE           35
INHABILITADO (Art. 5)     12
AUSENTE                    1

── Estadísticas de puntaje ──
count     290.000
mean      905.824
std       192.248
min       370.625
25%       752.969
50%       907.750
75%      1036.938
max      1434.750


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,338499,"Abarca Lapa, Deyvid Renzo",INGENIERÍA MECATRONICA,828.125,NaN,SIN OBSERVACIÓN,INGENIERÍA MECATRONICA,https://admision.unmsm.edu.pe/Website20262/A/1...
1,344362,"Acero Reyes, Arian Gerard Valentino",INGENIERÍA MECATRONICA,1171.500,24.0,ALCANZÓ VACANTE,INGENIERÍA MECATRONICA,https://admision.unmsm.edu.pe/Website20262/A/1...
2,354193,"Acuña Cabrera, Ronal",INGENIERÍA MECATRONICA,1031.625,NaN,SIN OBSERVACIÓN,INGENIERÍA MECATRONICA,https://admision.unmsm.edu.pe/Website20262/A/1...
3,321666,"Aguilar Becerra, Santiago Sebastian",INGENIERÍA MECATRONICA,974.000,NaN,SIN OBSERVACIÓN,INGENIERÍA MECATRONICA,https://admision.unmsm.edu.pe/Website20262/A/1...
4,340164,"Alarcon Ortiz, Daniel",INGENIERÍA MECATRONICA,1041.375,NaN,SIN OBSERVACIÓN,INGENIERÍA MECATRONICA,https://admision.unmsm.edu.pe/Website20262/A/1...
5,363193,"Alatrista Abad, Sonia Paola",INGENIERÍA MECATRONICA,868.375,NaN,SIN OBSERVACIÓN,INGENIERÍA MECATRONICA,https://admision.unmsm.edu.pe/Website20262/A/1...
6,324399,"Albites Padilla, Renzo Jairo",INGENIERÍA MECATRONICA,1020.250,NaN,SIN OBSERVACIÓN,INGENIERÍA MECATRONICA,https://admision.unmsm.edu.pe/Website20262/A/1...
7,382664,"Alfaro Monge, Kristofer Samir",INGENIERÍA MECATRONICA,948.875,NaN,SIN OBSERVACIÓN,INGENIERÍA MECATRONICA,https://admision.unmsm.edu.pe/Website20262/A/1...
8,332479,"Alvino Llanos, Leonardo Miguel",INGENIERÍA MECATRONICA,995.125,NaN,SIN OBSERVACIÓN,INGENIERÍA MECATRONICA,https://admision.unmsm.edu.pe/Website20262/A/1...
9,368289,"Angeles Gonzales, Alvaro",INGENIERÍA MECATRONICA,624.000,NaN,SIN OBSERVACIÓN,INGENIERÍA MECATRONICA,https://admision.unmsm.edu.pe/Website20262/A/1...


In [18]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_mecatronica.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERÍA MECATRONICA
Vacantes obtenidas: 35
Puntaje máximo: 1434.75
Puntaje mínimo: 1125.75


,codigo,apellidos_nombres,puntaje,merito_ep
0,321159,"Machaca Surco, Alvaro Manuel",1434.750,1.0
1,330048,"Zamata Rios, Leonard Rene",1399.875,2.0
2,351082,"Ortiz Palacio, José Alonso",1396.500,3.0
3,358030,"Puertas Olivares, Erick Daniel",1385.625,4.0
4,366451,"Millones Velasquez, Josue Miguel",1359.250,5.0
5,393455,"Vivanco Ccencho, Angel",1332.500,6.0
6,315338,"Baldeon Condori, Matt Calef",1329.125,7.0
7,331404,"Quiroz Cabanillas, André Aymar",1307.875,8.0
8,391477,"Encarnacion Polo, Harold Patrick",1294.875,9.0
9,348247,"Chura Callizana, Aaron Rodrigo",1273.750,10.0


In [19]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/165/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_minas.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/165/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 275 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_minas.csv
   275 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          240
ALCANZÓ VACANTE           25
AUSENTE                    5
INHABILITADO (Art. 5)      5

── Estadísticas de puntaje ──
count     270.000
mean      852.101
std       206.899
min       230.000
25%       703.844
50%       826.125
75%       996.062
max      1314.250


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,338269,"Acero Infante, Gabriel Adrian",INGENIERÍA DE MINAS,1046.500,NaN,SIN OBSERVACIÓN,INGENIERÍA DE MINAS,https://admision.unmsm.edu.pe/Website20262/A/1...
1,339663,"Aedo Pillaca, Juan Diego",INGENIERÍA DE MINAS,1139.500,NaN,SIN OBSERVACIÓN,INGENIERÍA DE MINAS,https://admision.unmsm.edu.pe/Website20262/A/1...
2,338773,"Aguilar Quispe, Marco Antonio",INGENIERÍA DE MINAS,814.125,NaN,SIN OBSERVACIÓN,INGENIERÍA DE MINAS,https://admision.unmsm.edu.pe/Website20262/A/1...
3,384482,"Aguirre Luna, Gerson Marcelo",INGENIERÍA DE MINAS,718.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE MINAS,https://admision.unmsm.edu.pe/Website20262/A/1...
4,335921,"Alata Lira, Deivid Alexander",INGENIERÍA DE MINAS,703.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE MINAS,https://admision.unmsm.edu.pe/Website20262/A/1...
5,399129,"Aldave Altamirano, Jeyson Fidel",INGENIERÍA DE MINAS,786.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE MINAS,https://admision.unmsm.edu.pe/Website20262/A/1...
6,312607,"Alejo Galindo, Yanlui Johan",INGENIERÍA DE MINAS,926.000,NaN,SIN OBSERVACIÓN,INGENIERÍA DE MINAS,https://admision.unmsm.edu.pe/Website20262/A/1...
7,370688,"Amaya Montesinos, Jurgen Gerard",INGENIERÍA DE MINAS,956.875,NaN,SIN OBSERVACIÓN,INGENIERÍA DE MINAS,https://admision.unmsm.edu.pe/Website20262/A/1...
8,386195,"Ambrosio Hermoza, Milagros Elizabeth",INGENIERÍA DE MINAS,673.750,NaN,SIN OBSERVACIÓN,INGENIERÍA DE MINAS,https://admision.unmsm.edu.pe/Website20262/A/1...
9,329909,"Aponte Sanchez, Hana Farah",INGENIERÍA DE MINAS,910.625,NaN,SIN OBSERVACIÓN,INGENIERÍA DE MINAS,https://admision.unmsm.edu.pe/Website20262/A/1...


In [20]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_minas.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERÍA DE MINAS
Vacantes obtenidas: 25
Puntaje máximo: 1314.25
Puntaje mínimo: 1175.375


,codigo,apellidos_nombres,puntaje,merito_ep
0,336488,"Contreras Palacios, Josue Nicolas",1314.250,1.0
1,346707,"Astocóndor Caldas, Fabrizzio Luciano",1311.875,2.0
2,380013,"Quispe Aniceto, Crhistopher Gabriel",1301.000,3.0
3,368883,"Romero Soto, Juan Alfonso",1294.875,4.0
4,325659,"Moreno Huamani, Aldo Santiago Joaquín",1280.000,5.0
5,341912,"Rocha Chavez, Jhordan",1273.125,6.0
6,322576,"Bazan Cabrera, Christopher Drake",1270.875,7.0
7,324443,"Roncal Millan, Diego Anibal",1269.750,8.0
8,365235,"Ortiz Mostacero, Jose Cristian Del Piero",1257.125,9.0
9,355689,"Garcia Rodriguez, Davis Jostin",1254.875,10.0


In [21]:
import time
import base64
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# ══════════════════════════════════════════════
# ▶ CONFIGURACIÓN
# ══════════════════════════════════════════════

URL            = "https://admision.unmsm.edu.pe/Website20262/A/202/results.html"
BATCH_RUTAS    = []   # ejemplo: ["A/091", "A/092"]
ARCHIVO_SALIDA = "resultados_unmsm_software.csv"
BASE_URL       = "https://admision.unmsm.edu.pe/Website20262"
WAIT_SEGUNDOS  = 8

# ══════════════════════════════════════════════


def crear_driver():
    opciones = Options()
    opciones.add_argument("--headless=new")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-gpu")
    opciones.add_argument("--window-size=1920,1080")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def mostrar_todas_las_filas(driver):
    """
    Intenta mostrar TODOS los registros de una vez usando 3 estrategias:
    1. Cambiar el select de longitud de DataTables a -1 (All) vía Selenium
    2. Forzarlo por JavaScript directamente sobre la instancia DataTable
    3. Si no hay selector, navegar página por página
    """
    wait = WebDriverWait(driver, 10)

    # ── Estrategia 1: select de longitud (típico en DataTables) ──────────
    try:
        select_el = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "select[name*='length'], select[name*='DataTables']"))
        )
        select = Select(select_el)
        try:
            select.select_by_value("-1")
            print("  [✓] Paginación: seleccionado 'Todos' en el select.")
        except Exception:
            opciones_vals = [o.get_attribute("value") for o in select.options]
            select.select_by_value(opciones_vals[-1])
            print(f"  [✓] Paginación: seleccionado valor máximo ({opciones_vals[-1]}) en el select.")
        time.sleep(3)
        return "select"
    except Exception:
        pass

    # ── Estrategia 2: forzar via JavaScript DataTables API ───────────────
    try:
        driver.execute_script("""
            var tables = $.fn.dataTable ? $.fn.dataTable.tables() : [];
            if (tables.length > 0) {
                $(tables[0]).DataTable().page.len(-1).draw();
            }
        """)
        time.sleep(3)
        print("  [✓] Paginación: forzado via DataTables JS API.")
        return "js"
    except Exception:
        pass

    # ── Estrategia 3: paginación manual (clic en "Siguiente") ────────────
    print("  [!] Usando paginación manual página por página...")
    return "manual"


def obtener_html_completo(driver, url: str) -> str:
    """Carga la página y retorna el HTML con TODOS los registros visibles."""
    driver.get(url)
    time.sleep(WAIT_SEGUNDOS)

    modo = mostrar_todas_las_filas(driver)

    if modo == "manual":
        return obtener_html_paginado(driver)

    return driver.page_source


def obtener_html_paginado(driver) -> str:
    """
    Recorre todas las páginas haciendo clic en 'Siguiente'
    y acumula los <tr> de cada página en una tabla unificada.
    """
    todas_filas_html = []
    pagina = 1

    while True:
        soup  = BeautifulSoup(driver.page_source, "html.parser")
        tabla = soup.find("table")
        if tabla:
            filas = tabla.find_all("tr")[1:]  # sin encabezado
            todas_filas_html.extend([str(f) for f in filas])
            print(f"    Página {pagina}: {len(filas)} filas | total acumulado: {len(todas_filas_html)}")

        try:
            siguiente = driver.find_element(
                By.CSS_SELECTOR,
                "a.paginate_button.next:not(.disabled), button.paginate_button.next:not(.disabled)"
            )
            siguiente.click()
            pagina += 1
            time.sleep(2)
        except Exception:
            print(f"    Fin de paginación en página {pagina}.")
            break

    # Reconstruir HTML con todas las filas
    soup_base  = BeautifulSoup(driver.page_source, "html.parser")
    tabla_base = soup_base.find("table")
    if tabla_base:
        tbody = tabla_base.find("tbody")
        if tbody:
            tbody.clear()
            for fila_html in todas_filas_html:
                tbody.append(BeautifulSoup(fila_html, "html.parser"))
    return str(soup_base)


def decodificar_b64(valor: str) -> str:
    try:
        return base64.b64decode(valor).decode("utf-8").strip()
    except Exception:
        return valor.strip()


def extraer_carrera(soup: BeautifulSoup) -> str:
    items = soup.select("ol li, ul.breadcrumb li")
    if items:
        return items[-1].get_text(strip=True)
    h1 = soup.find("h1")
    return h1.get_text(strip=True) if h1 else "DESCONOCIDA"


def normalizar_obs(val: str) -> str:
    v = val.strip().upper()
    if "ALCANZ" in v and "VACANTE" in v:
        return "ALCANZÓ VACANTE"
    if "AUSENTE" in v:
        return "AUSENTE"
    if "ART" in v:
        return "INHABILITADO (Art. 5)"
    return v if v else "SIN OBSERVACIÓN"


def parse_tabla(html: str, url: str) -> pd.DataFrame:
    soup  = BeautifulSoup(html, "html.parser")
    tabla = soup.find("table")

    if tabla is None:
        print(f"[AVISO] No se encontró tabla en: {url}")
        return pd.DataFrame()

    filas = []
    for tr in tabla.find_all("tr")[1:]:
        celdas = tr.find_all("td")
        if not celdas:
            continue

        codigo  = celdas[0].get_text(strip=True) if len(celdas) > 0 else ""

        nombre = ""
        if len(celdas) > 1:
            span = celdas[1].find("span", class_="obfuscated")
            nombre = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[1].get_text(strip=True)

        escuela = ""
        if len(celdas) > 2:
            span = celdas[2].find("span", class_="obfuscated")
            escuela = decodificar_b64(span["data-auth"]) if (span and span.get("data-auth")) else celdas[2].get_text(strip=True)

        puntaje = celdas[3].get("data-score", "").strip() if len(celdas) > 3 else ""
        merito  = celdas[4].get("data-merit", "").strip() if len(celdas) > 4 else ""
        obs     = celdas[5].get_text(strip=True)          if len(celdas) > 5 else ""

        filas.append({
            "codigo":            codigo,
            "apellidos_nombres": nombre,
            "escuela":           escuela,
            "puntaje":           puntaje,
            "merito_ep":         merito,
            "observacion":       obs,
        })

    if not filas:
        print(f"[AVISO] Sin filas en: {url}")
        return pd.DataFrame()

    df = pd.DataFrame(filas)
    df["codigo"] = df["codigo"].str.strip()
    df["apellidos_nombres"] = (
        df["apellidos_nombres"].str.strip().str.title()
        .str.replace(r"\s{2,}", " ", regex=True)
    )
    df["puntaje"]    = pd.to_numeric(df["puntaje"],  errors="coerce")
    df["merito_ep"]  = pd.to_numeric(df["merito_ep"], errors="coerce")
    df["observacion"] = df["observacion"].apply(normalizar_obs)
    df["carrera"]    = extraer_carrera(soup)
    df["url_fuente"] = url

    df = df.dropna(how="all").drop_duplicates(subset=["codigo"]).reset_index(drop=True)
    return df


def scrape_url(driver, url: str) -> pd.DataFrame:
    print(f"  Abriendo: {url}")
    html = obtener_html_completo(driver, url)
    df   = parse_tabla(html, url)
    print(f"  → {len(df)} registros extraídos.")
    return df


def guardar_csv(df: pd.DataFrame, ruta: Path):
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV guardado: {ruta.resolve()}")
    print(f"   {len(df)} filas  ×  {len(df.columns)} columnas")
    print("\n── Observaciones ──")
    print(df["observacion"].value_counts().to_string())
    if df["puntaje"].notna().any():
        print("\n── Estadísticas de puntaje ──")
        print(df["puntaje"].describe().round(3).to_string())


# ══════════════════════════════════════════════
# ▶ EJECUCIÓN
# ══════════════════════════════════════════════

driver = crear_driver()

try:
    if BATCH_RUTAS:
        dfs = []
        for ruta in BATCH_RUTAS:
            url = f"{BASE_URL}/{ruta.strip('/')}/results.html"
            df  = scrape_url(driver, url)
            if not df.empty:
                dfs.append(df)
            time.sleep(1.5)
        resultado = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        resultado = scrape_url(driver, URL)
finally:
    driver.quit()
    print("[INFO] Navegador cerrado.")

if not resultado.empty:
    guardar_csv(resultado, Path(ARCHIVO_SALIDA))
    display(resultado.head(15))
else:
    print("❌ No se obtuvieron datos.")

  Abriendo: https://admision.unmsm.edu.pe/Website20262/A/202/results.html
  [✓] Paginación: forzado via DataTables JS API.
  → 411 registros extraídos.
[INFO] Navegador cerrado.

✅ CSV guardado: C:\Users\confe\OneDrive\Documentos\ANÁLISIS DE DATOS UNMSM 2026\resultados_unmsm_software.csv
   411 filas  ×  8 columnas

── Observaciones ──
observacion
SIN OBSERVACIÓN          352
ALCANZÓ VACANTE           49
AUSENTE                    5
INHABILITADO (Art. 5)      5

── Estadísticas de puntaje ──
count     406.000
mean      903.069
std       227.601
min       313.875
25%       739.281
50%       878.000
75%      1059.469
max      1502.125


,codigo,apellidos_nombres,escuela,puntaje,merito_ep,observacion,carrera,url_fuente
0,374216,"Abad Maldonado, Merly Edith",INGENIERÍA DE SOFTWARE,667.250,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SOFTWARE,https://admision.unmsm.edu.pe/Website20262/A/2...
1,363985,"Acosta Llacua, Clever",INGENIERÍA DE SOFTWARE,743.875,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SOFTWARE,https://admision.unmsm.edu.pe/Website20262/A/2...
2,385593,"Adan Sandoval, John Jairo",INGENIERÍA DE SOFTWARE,962.000,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SOFTWARE,https://admision.unmsm.edu.pe/Website20262/A/2...
3,344661,"Aguilar Noa, Guillermo André",INGENIERÍA DE SOFTWARE,720.500,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SOFTWARE,https://admision.unmsm.edu.pe/Website20262/A/2...
4,357970,"Aguilar Palomino, Gabriel Angel Ygnacio",INGENIERÍA DE SOFTWARE,612.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SOFTWARE,https://admision.unmsm.edu.pe/Website20262/A/2...
5,353008,"Ahumada Maldonado, Michael Anderson",INGENIERÍA DE SOFTWARE,740.875,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SOFTWARE,https://admision.unmsm.edu.pe/Website20262/A/2...
6,394153,"Alcántara Reyes, Kimberly Camila",INGENIERÍA DE SOFTWARE,NaN,NaN,AUSENTE,INGENIERÍA DE SOFTWARE,https://admision.unmsm.edu.pe/Website20262/A/2...
7,341366,"Aldave Yarleque, Fabrizio",INGENIERÍA DE SOFTWARE,703.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SOFTWARE,https://admision.unmsm.edu.pe/Website20262/A/2...
8,346527,"Aldazabal Quijada, Jean Paul",INGENIERÍA DE SOFTWARE,1130.375,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SOFTWARE,https://admision.unmsm.edu.pe/Website20262/A/2...
9,360584,"Alegria Rojas, Jaren Jairo",INGENIERÍA DE SOFTWARE,1000.750,NaN,SIN OBSERVACIÓN,INGENIERÍA DE SOFTWARE,https://admision.unmsm.edu.pe/Website20262/A/2...


In [22]:
import pandas as pd

df = pd.read_csv("resultados_unmsm_software.csv")

vacantes = df[df["observacion"] == "ALCANZÓ VACANTE"]

max_p = vacantes["puntaje"].max()
min_p = vacantes["puntaje"].min()

print(f"Carrera: {df['carrera'].iloc[0]}")
print(f"Vacantes obtenidas: {len(vacantes)}")
print(f"Puntaje máximo: {max_p}")
print(f"Puntaje mínimo: {min_p}")

display(vacantes[["codigo","apellidos_nombres","puntaje","merito_ep"]]
        .sort_values("puntaje", ascending=False)
        .reset_index(drop=True))

Carrera: INGENIERÍA DE SOFTWARE
Vacantes obtenidas: 49
Puntaje máximo: 1502.125
Puntaje mínimo: 1177.75


,codigo,apellidos_nombres,puntaje,merito_ep
0,382350,"Teves Chamorro, Andre Samuel",1502.125,1.0
1,356163,"Falconi Tabata, Elder Andhre Kazuo",1498.125,2.0
2,377483,"Espinoza Bautista, Bertolt Andrey",1495.250,3.0
3,378282,"Cespedes Reyes, Andre Raul",1471.875,4.0
4,314052,"Rivera Vega, Helton Jhon",1466.125,5.0
5,361748,"Koo Saenz, Alexandra Mia",1463.750,6.0
6,322011,"Loayza Pajuelo, Yazmin Yessú",1449.000,7.0
7,357098,"Bueno Luyo, Aaron Americo Fernando",1438.750,8.0
8,323520,"Huertas Balcázar, Jaquelin Alexandra",1418.750,9.0
9,330664,"Arias Mendoza, Jesu Alejandro",1417.625,10.0
